# Pipeline ETL Completo — Sinistros de Trânsito (SP, 2021–2023)

- **Tarefa:** ETL-26 · Notebook ETL completo, rodado do zero
- **Responsável:** Guilherme Calistrato (`@guicalistrato`) | **Revisor:** Davi Mello (`@davi01mello`)
- **Fonte:** 36 arquivos CSV mensais em `data/pessoas_MM-AAAA.csv` (806.977 registros)
- **Destino:** Esquema estrela (`fato_pessoa_sinistro` + 8 dimensões) no PostgreSQL (Supabase)
- **Geração automática:** este notebook foi montado a partir das etapas em `notebooks/etl/` pelo script `scripts/montar_notebook_etl.py`, com as entradas encadeadas ponta a ponta.

## Visão geral do encadeamento

| Ordem | Etapa | Entrada | Saída |
| --- | --- | --- | --- |
| 01 | Extração (`ETL-05`) | `data/pessoas_*.csv` | `01_bruto.parquet` |
| 02 | Chaves e duplicatas (`ETL-13`) | `01_bruto.parquet` | `02_chaves_duplicatas.parquet` |
| 03 | Texto e encoding (`ETL-14`) | `02_chaves_duplicatas.parquet` | `03_texto.parquet` |
| 04 | Datas (`ETL-15`) | `03_texto.parquet` | `04_datas.parquet` |
| 05 | Nulos e categorias (`ETL-16`) | `04_datas.parquet` | `05_nulos_categorias.parquet` |
| 06 | Profissão (`ETL-17`) | `05_nulos_categorias.parquet` | `06_limpo.parquet` |
| 07a | `dim_tempo` e `dim_gravidade` (`ETL-19`) | `06_limpo.parquet` | `dims/dim_tempo.parquet`, `dims/dim_gravidade.parquet` |
| 07b | `dim_perfil_vitima` e `dim_transporte` (`ETL-20`) | `06_limpo.parquet` | `dims/dim_perfil_vitima.parquet`, `dims/dim_transporte.parquet` |
| 07c | `dim_localidade` e `dim_via` (`ETL-21`) | `06_limpo.parquet` | `dims/dim_localidade.parquet`, `dims/dim_via.parquet` |
| 07d | `dim_profissao` e `dim_local_obito` (`ETL-22`) | `06_limpo.parquet` | `dims/dim_profissao.parquet`, `dims/dim_local_obito.parquet` |
| 08 | Tabela fato (`ETL-23`) | `06_limpo.parquet` + `dims/*.parquet` | `fato.parquet` |
| 09 | Carga no PostgreSQL (`ETL-24`) | `sql/ddl/etl_star_schema.sql` + parquets | 9 tabelas no schema `public` |
| 10 | Validação pós-carga (`ETL-25`) | Banco PostgreSQL | `sql/validacao/etl_checks.sql` |

## Setup Global

In [1]:
import io
import os
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd
from IPython.display import display

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)

if str(RAIZ / "src") not in sys.path:
    sys.path.append(str(RAIZ / "src"))

from conexao import obter_engine, testar_conexao
from perfil import _p4_texto

DADOS  = RAIZ / "data"
INTERM = DADOS / "intermediarios"
DIMS   = INTERM / "dims"
INTERM.mkdir(parents=True, exist_ok=True)
DIMS.mkdir(parents=True, exist_ok=True)


def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_nulo": (df.isna().mean() * 100).round(2),
        "n_distintos": df.nunique(),
    })


print("RAIZ:", RAIZ)

RAIZ: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD


---

## Etapa 01 · Extração dos 36 CSVs (ETL-05)

- **Etapa:** ETL-05, Extração dos 36 CSVs
- **Responsável:** Pedro Wellington
- **Issue:** ETL-05
- **Entrada:** `data/pessoas_*.csv`
- **Saída:** `data/intermediarios/01_bruto.parquet`

Lê os 36 CSVs mensais (2021 a 2023), junta numa tabela só e salva em parquet. Nenhum dado é alterado.

### 1. Setup

In [2]:
SAIDA = INTERM / "01_bruto.parquet"            # resultado desta etapa

### 2. Carregar a entrada

In [3]:
arquivos = sorted(DADOS.glob("pessoas_*.csv"))

lista = []
for a in arquivos:
    d = pd.read_csv(a, encoding='latin-1', sep=';')
    d['ano_origem'] = int(a.stem[-4:])
    d['arquivo_origem'] = a.name
    lista.append(d)

df = pd.concat(lista, ignore_index=True)
print(len(arquivos), 'arquivos'); print('shape:', df.shape); df.head()

36 arquivos
shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


### 3. Diagnóstico ANTES

In [4]:
antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


### 4. Conferências

In [5]:
# colunas originais: as mesmas, na mesma ordem, nos 36 arquivos
base = list(lista[0].columns[:-2])
diferentes = [a.name for a, d in zip(arquivos, lista) if list(d.columns[:-2]) != base]
print(len(base), 'colunas;', len(diferentes), 'arquivos com colunas diferentes')

# total de linhas
print('total de linhas:', len(df))
assert len(df) == 806977

31 colunas; 0 arquivos com colunas diferentes
total de linhas: 806977


### 5.  Salvar a saída

In [6]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\01_bruto.parquet | shape: (806977, 33)


### 6. Resumo

**O que foi feito:** leitura dos 36 arquivos mensais (2021 a 2023) e junção em uma única tabela de 806.977 linhas. Nenhum dado foi alterado. O resultado foi salvo em `data/intermediarios/01_bruto.parquet`.

**Por quê:** as etapas seguintes leem essa tabela, em vez de abrir os 36 CSVs de novo. O parquet ocupa menos espaço e carrega mais rápido.

**O que foi conferido:**
- os 36 arquivos têm as mesmas 31 colunas, na mesma ordem
- o total é de 806.977 linhas

**O que foi acrescentado:** `ano_origem` e `arquivo_origem`, para saber de qual arquivo cada linha veio. A tabela final tem 33 colunas.

**Observação:** o `.parquet` não vai para o Git. Quem precisar dele roda este notebook.

---

## Etapa 02 · Chaves e duplicatas (ETL-13)

- **Etapa:** 02, chaves e duplicatas
- **Responsável:** Davi (@davi01mello)
- **Issue:** #15
- **Entrada:** `intermediarios/01_bruto.parquet` (saída da ETL-05)
- **Saída:** `intermediarios/02_chaves_duplicatas.parquet`

**O que esta etapa garante para o modelo estrela (ETL-10):**
1. Os IDs (`id_sinistro`, `id_pessoa`, `id_veiculo`) e o `cod_ibge` viram inteiros, sem perder nenhum valor.
2. Não há linhas 100% duplicadas. Se houver, são removidas e contadas.
3. `id_pessoa` é único, porque ele é a chave primária da fato (1 linha = 1 pessoa em 1 sinistro).
4. Cada `cod_ibge` tem um município só, porque ele é a chave natural da `dim_localidade`.

### 1. Setup

In [7]:
ENTRADA = INTERM / "01_bruto.parquet"       # na execução final, a saída da etapa anterior
SAIDA   = INTERM / "02_chaves_duplicatas.parquet"

COLUNAS_ID = ["id_sinistro", "id_pessoa", "id_veiculo", "cod_ibge"]
COLUNAS_RASTREIO = ["ano_origem", "arquivo_origem"]   # criadas na extração, não vêm da fonte

### 2. Carregar a entrada

Lê o `01_bruto.parquet` da ETL-05. **Enquanto ele não existir**, lê os 36 CSVs do mesmo jeito que a extração deve ler (tudo como texto, `latin-1`, `;`) e avisa. Quando a ETL-05 entrar na `main`, este bloco passa a usar o parquet sem nenhuma mudança.

In [8]:
def ler_csvs_brutos(pasta: Path) -> pd.DataFrame:
    """Provisório até a ETL-05: lê os 36 CSVs como texto e cria as colunas de rastreio."""
    partes = []
    for arq in sorted(pasta.glob("pessoas_*.csv")):
        d = pd.read_csv(arq, sep=";", encoding="latin-1", dtype=str,
                        keep_default_na=False, na_values=[""])
        d["ano_origem"] = int(arq.stem[-4:])
        d["arquivo_origem"] = arq.name
        partes.append(d)
    return pd.concat(partes, ignore_index=True)

if ENTRADA.exists():
    df = pd.read_parquet(ENTRADA)
    print("entrada:", ENTRADA.name)
else:
    df = ler_csvs_brutos(DADOS)
    print("AVISO: 01_bruto.parquet não existe ainda; lendo os CSVs direto (provisório até a ETL-05)")

print("shape:", df.shape)
df[COLUNAS_ID].head()

entrada: 01_bruto.parquet
shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge
0,2240338,1979797,2874666.0,3552502
1,2240338,1979798,2874666.0,3552502
2,1447800,1284984,NaN,3550308
3,1783410,563279,2648403.0,3550308
4,1798856,687322,NaN,3547601


### 3. Diagnóstico ANTES

In [9]:
def diagnostico_chaves(df: pd.DataFrame) -> pd.Series:
    """Os números que esta etapa precisa garantir, num quadro só (usado antes e depois)."""
    colunas_fonte = [c for c in df.columns if c not in COLUNAS_RASTREIO]
    return pd.Series({
        "linhas": len(df),
        "linhas 100% duplicadas (31 colunas da fonte)": int(df.duplicated(subset=colunas_fonte).sum()),
        "id_pessoa repetidos": int(df["id_pessoa"].duplicated().sum()),
        "id_pessoa nulos": int(df["id_pessoa"].isna().sum()),
        "id_sinistro nulos": int(df["id_sinistro"].isna().sum()),
        "id_veiculo nulos": int(df["id_veiculo"].isna().sum()),
        "sinistros distintos": int(df["id_sinistro"].nunique()),
        "cod_ibge com mais de 1 município": int((df.groupby("cod_ibge")["municipio"].nunique() > 1).sum()),
        "município com mais de 1 cod_ibge": int((df.groupby("municipio")["cod_ibge"].nunique() > 1).sum()),
    })

antes = diagnostico(df)
chaves_antes = diagnostico_chaves(df)
display(antes.loc[COLUNAS_ID])
chaves_antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645


linhas                                          806977
linhas 100% duplicadas (31 colunas da fonte)         0
id_pessoa repetidos                                  0
id_pessoa nulos                                      0
id_sinistro nulos                                    0
id_veiculo nulos                                169754
sinistros distintos                             402695
cod_ibge com mais de 1 município                     0
município com mais de 1 cod_ibge                     0
dtype: int64

### 4. Transformações

#### 4.1 IDs e `cod_ibge` para inteiro

`id_sinistro`, `id_pessoa` e `cod_ibge` nunca são nulos e viram `int64`. `id_veiculo` é nulo em cerca de 21% das linhas (pedestres e registros sem veículo) e vira `Int64`, o inteiro do pandas que aceita nulo. Sem isso ele viraria `float` (`123.0`).

A conversão usa `errors="raise"`: se algum valor não for número, o notebook para em vez de virar nulo em silêncio. Depois a função confere que nenhum nulo novo apareceu.

In [10]:
def converter_ids(df: pd.DataFrame) -> pd.DataFrame:
    """Converte IDs e cod_ibge para inteiro; falha se algum valor não for numérico ou se surgir nulo novo."""
    df = df.copy()
    nulos_antes = df[COLUNAS_ID].isna().sum()
    for col in ["id_sinistro", "id_pessoa", "cod_ibge"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")
    df["id_veiculo"] = pd.to_numeric(df["id_veiculo"], errors="raise").astype("Int64")
    nulos_depois = df[COLUNAS_ID].isna().sum()
    assert (nulos_antes == nulos_depois).all(), "a conversão criou nulos novos"
    return df

df = converter_ids(df)
print(df[COLUNAS_ID].dtypes)
print("nulos por coluna (iguais aos de antes):")
print(df[COLUNAS_ID].isna().sum())

id_sinistro    int64
id_pessoa      int64
id_veiculo     Int64
cod_ibge       int64
dtype: object
nulos por coluna (iguais aos de antes):
id_sinistro         0
id_pessoa           0
id_veiculo     169754
cod_ibge            0
dtype: int64


#### 4.2 Linhas 100% duplicadas

A comparação usa **as 31 colunas da fonte**, sem `ano_origem` e `arquivo_origem`. Assim, uma mesma linha que aparecesse em dois arquivos mensais também contaria como duplicata. Fica a primeira ocorrência.

In [11]:
def remover_duplicatas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove linhas idênticas nas 31 colunas da fonte, mantendo a primeira ocorrência."""
    colunas_fonte = [c for c in df.columns if c not in COLUNAS_RASTREIO]
    return df.drop_duplicates(subset=colunas_fonte, keep="first").reset_index(drop=True)

n_antes = len(df)
df = remover_duplicatas(df)
print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)}, motivo: linha 100% duplicada)")

linhas: 806977 -> 806977 (removidas: 0, motivo: linha 100% duplicada)


#### 4.3 `id_pessoa` único

É a chave primária da fato. Se algum `id_pessoa` aparecer duas vezes com conteúdo diferente, o notebook para e mostra os casos, porque escolher uma das linhas seria uma decisão do grupo, não desta etapa.

In [12]:
def garantir_id_pessoa_unico(df: pd.DataFrame) -> pd.DataFrame:
    """Confere que id_pessoa é único; se não for, mostra os casos e interrompe."""
    repetidos = df[df["id_pessoa"].duplicated(keep=False)].sort_values("id_pessoa")
    if len(repetidos):
        display(repetidos)
        raise ValueError(f"{repetidos['id_pessoa'].nunique()} id_pessoa repetidos com conteúdo diferente")
    print(f"id_pessoa único: {df['id_pessoa'].nunique():,} valores em {len(df):,} linhas".replace(",", "."))
    return df

df = garantir_id_pessoa_unico(df)

id_pessoa único: 806.977 valores em 806.977 linhas


#### 4.4 `cod_ibge` → município

Cada código IBGE deve ter um município só, e vice-versa, porque o `cod_ibge` é a chave natural da `dim_localidade`. Esta conferência também olha a região administrativa, já que cada município deve pertencer a uma região só.

In [13]:
def conferir_localidade(df: pd.DataFrame) -> pd.DataFrame:
    """Confere cod_ibge 1:1 com municipio e 1 região por município; devolve a tabela de municípios."""
    loc = df.groupby("cod_ibge").agg(
        municipios=("municipio", "nunique"),
        regioes=("regiao_administrativa", "nunique"),
        municipio=("municipio", "first"),
        regiao_administrativa=("regiao_administrativa", "first"),
        linhas=("id_pessoa", "size"),
    )
    problemas = loc[(loc["municipios"] > 1) | (loc["regioes"] > 1)]
    assert problemas.empty, f"{len(problemas)} cod_ibge com mais de um município ou região"
    assert df.groupby("municipio")["cod_ibge"].nunique().max() == 1, "município com mais de um cod_ibge"
    print(f"{len(loc)} códigos IBGE, {df['municipio'].nunique()} municípios, "
          f"{df['regiao_administrativa'].nunique()} regiões: relação 1:1 confirmada")
    return loc

localidades = conferir_localidade(df)
localidades.sort_values("linhas", ascending=False).head(10)

645 códigos IBGE, 645 municípios, 16 regiões: relação 1:1 confirmada


,municipios,regioes,municipio,regiao_administrativa,linhas
cod_ibge,,,,,
3550308,1,1,SAO PAULO,METROPOLITANA DE SÃO PAULO,186723
3543402,1,1,RIBEIRAO PRETO,RIBEIRÃO PRETO,21395
3509502,1,1,CAMPINAS,CAMPINAS,20429
3518800,1,1,GUARULHOS,METROPOLITANA DE SÃO PAULO,19452
3549805,1,1,SAO JOSE DO RIO PRETO,SÃO JOSÉ DO RIO PRETO,16521
3552205,1,1,SOROCABA,SOROCABA,14786
3548708,1,1,SAO BERNARDO DO CAMPO,METROPOLITANA DE SÃO PAULO,14494
3547809,1,1,SANTO ANDRE,METROPOLITANA DE SÃO PAULO,11767
3525904,1,1,JUNDIAI,CAMPINAS,11090


#### 4.5 Conferências extras de integridade (só consulta)

Três perguntas que a ETL-23 (fato) vai precisar que sejam verdade:
- um veículo (`id_veiculo`) pertence a um sinistro só?
- um sinistro tem uma data e um município só?
- há pessoas com **tudo igual menos o `id_pessoa`**? Não são duplicatas (são pessoas diferentes), mas vale saber quantas são.

In [14]:
veiculo_em_varios = (df.dropna(subset=["id_veiculo"]).groupby("id_veiculo")["id_sinistro"].nunique() > 1).sum()
sinistro_varias_datas = (df.groupby("id_sinistro")["data_sinistro"].nunique() > 1).sum()
sinistro_varios_municipios = (df.groupby("id_sinistro")["cod_ibge"].nunique() > 1).sum()

colunas_sem_id = [c for c in df.columns if c not in ["id_pessoa", *COLUNAS_RASTREIO]]
quase = df[df.duplicated(subset=colunas_sem_id, keep=False)]

print("veículos em mais de 1 sinistro:", veiculo_em_varios)
print("sinistros com mais de 1 data:", sinistro_varias_datas)
print("sinistros com mais de 1 município:", sinistro_varios_municipios)
print(f"linhas iguais exceto id_pessoa: {len(quase)} em "
      f"{quase.groupby(colunas_sem_id, dropna=False).ngroups} grupos")
quase["tipo_de_vitima"].value_counts()

veículos em mais de 1 sinistro: 0
sinistros com mais de 1 data: 0
sinistros com mais de 1 município: 0
linhas iguais exceto id_pessoa: 1211 em 571 grupos


tipo_de_vitima
NAO DISPONIVEL    643
PASSAGEIRO        470
PEDESTRE           88
CONDUTOR           10
Name: count, dtype: int64

### 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [15]:
depois = diagnostico(df)
chaves_depois = diagnostico_chaves(df)
display(depois.loc[COLUNAS_ID])
pd.DataFrame({"antes": chaves_antes, "depois": chaves_depois})

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645


,antes,depois
linhas,806977,806977
linhas 100% duplicadas (31 colunas da fonte),0,0
id_pessoa repetidos,0,0
id_pessoa nulos,0,0
id_sinistro nulos,0,0
id_veiculo nulos,169754,169754
sinistros distintos,402695,402695
cod_ibge com mais de 1 município,0,0
município com mais de 1 cod_ibge,0,0


### 6. Salvar a saída

In [16]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA.relative_to(RAIZ), '| shape:', df.shape)

salvo em: data\intermediarios\02_chaves_duplicatas.parquet | shape: (806977, 33)


### 7. Resumo

- **IDs para inteiro:** `id_sinistro`, `id_pessoa` e `cod_ibge` viraram `int64`, e `id_veiculo` virou `Int64` (aceita nulo). Nenhum valor deixou de ser número e nenhum nulo novo apareceu. Os 21% de `id_veiculo` nulos continuam nulos, porque são pedestres e registros sem veículo.
- **Duplicatas:** **0** linhas 100% duplicadas nas 31 colunas da fonte, então nenhuma linha foi removida. A base sai com as mesmas 806.977 linhas.
- **`id_pessoa`:** é único nas 806.977 linhas e pode ser a chave primária da fato.
- **`cod_ibge`:** 645 códigos para 645 municípios (1:1), cada um em uma única das 16 regiões. A chave natural da `dim_localidade` é segura.
- **Integridade extra:** nenhum veículo aparece em mais de um sinistro, e cada sinistro tem uma data e um município só.
- **Achado:** 1.211 linhas (571 grupos) têm tudo igual exceto o `id_pessoa`. Não foram removidas, porque são pessoas diferentes no mesmo sinistro (a maioria passageiros ou registros com tipo de vítima "NAO DISPONIVEL"). O fato é registrado para a análise saber que existem.

---

## Etapa 03 · Texto e encoding (ETL-14)

- **Etapa:** `03_texto_encoding` (ETL-14)
- **Responsável:** Guilherme Calistrato (`@guicalistrato`)
- **Revisor:** Davi Mello (`@davi01mello`)
- **Issue:** #14 (ETL-14)
- **Entrada:** `data/intermediarios/02_chaves_duplicatas.parquet`
- **Saída:** `data/intermediarios/03_texto.parquet`

### 1. Setup

In [17]:
ENTRADA = INTERM / "02_chaves_duplicatas.parquet"
SAIDA   = INTERM / "03_texto.parquet"

# Colunas de identificação, datas, métricas e rastreio que não passam pela limpeza textual
COLUNAS_NAO_TEXTO = {
    "id_sinistro", "id_pessoa", "id_veiculo", "cod_ibge", "idade",
    "data_sinistro", "ano_sinistro", "mes_sinistro", "dia_sinistro", "ano_mes_sinistro",
    "data_obito", "ano_obito", "mes_obito", "dia_obito", "ano_mes_obito",
    "tempo_sinistro_obito", "ano_origem", "arquivo_origem",
}

### 2. Carregar a entrada

In [18]:
df = pd.read_parquet(ENTRADA)
print("shape:", df.shape)
df.head()

shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,<NA>,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,<NA>,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


### 3. Diagnóstico ANTES

In [19]:
def listar_colunas_texto(df: pd.DataFrame) -> list[str]:
    """Seleciona as colunas textuais/categóricas do DataFrame, excluindo IDs, datas e rastreio."""
    return [
        c for c in df.columns
        if c not in COLUNAS_NAO_TEXTO
        and (pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c]))
    ]

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


In [20]:
colunas_texto = listar_colunas_texto(df)
det_antes, res_antes = _p4_texto(df[colunas_texto])

print(f"Colunas de texto avaliadas ({len(colunas_texto)}):", colunas_texto)
print("\nResumo de problemas detectados por src/perfil.py (_p4_texto):")
display(res_antes)

print("\nDetalhe dos problemas (excluindo nacionalidade com espaço nas pontas):")
display(det_antes[det_antes["coluna"] != "nacionalidade"])

municipios_sem_apostrofo = (
    df[df["municipio"].str.contains(r"\bD [AO]", regex=True, na=False)]["municipio"]
    .value_counts()
    .rename_axis("municipio")
    .reset_index(name="linhas")
)
print("\nMunicípios com apóstrofo substituído por espaço (D OESTE / D ALHO):")
display(municipios_sem_apostrofo)

Colunas de texto avaliadas (15): ['municipio', 'regiao_administrativa', 'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima', 'sexo', 'gravidade_lesao', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'profissao', 'grau_de_instrucao', 'nacionalidade', 'local_obito', 'local_via']

Resumo de problemas detectados por src/perfil.py (_p4_texto):


,coluna,motivo,valores_distintos,linhas
0,profissao,espaco_duplo,1,1
1,profissao,letra_isolada,1,1
2,nacionalidade,espaco_nas_pontas,90,711291



Detalhe dos problemas (excluindo nacionalidade com espaço nas pontas):


,coluna,valor,n,motivo
0,profissao,TECNICO EM MECANICA PRODUCAO,1,espaco_duplo
1,profissao,Tecnico em raio x,1,letra_isolada



Municípios com apóstrofo substituído por espaço (D OESTE / D ALHO):


,municipio,linhas
0,SANTA BARBARA D OESTE,4111
1,PALMEIRA D OESTE,231
2,ESTRELA D OESTE,161
3,APARECIDA D OESTE,83
4,SANTA RITA D OESTE,36
5,SAO JOAO DO PAU D ALHO,18
6,SANTA CLARA D OESTE,15
7,GUARANI D OESTE,10


### 4. Transformações

Cada transformação é uma função com docstring. Nenhuma linha deve ser removida nesta etapa.

**Decisões de escopo adotadas:**
1. **Apóstrofo nos municípios (`D OESTE` / `D ALHO`):** Os 8 municípios paulistas que perderam o apóstrofo na fonte (ex.: `SANTA BARBARA D OESTE`, `SAO JOAO DO PAU D ALHO`) são padronizados **com apóstrofo** (`SANTA BARBARA D'OESTE`, `SAO JOAO DO PAU D'ALHO`), mantendo a ausência de acentos alinhada aos demais 637 municípios da coluna `municipio`.
2. **Acentuação e encoding:** Na leitura correta dos CSVs em `latin-1`, nenhum caractere inválido (`U+FFFD`) ou *mojibake* ocorre na base; ainda assim, o `MAPA_TEXTO` inclui entradas preventivas para os casos observados em amostras anteriores (`METROPOLITANA DE S O PAULO`, `M DIO`), garantindo proteção caso o pipeline receba extrações com perda de acento.
3. **Caixa alta/baixa em `profissao` (delegada para a ETL-17):** Foram identificados 266 valores distintos de `profissao` em caixa mista/minúscula (ex.: `'Motorista'`, `'Estudante'`, `'Tecnico em raio x'`) coexistindo com grafias em maiúsculas (`'MOTORISTA'`, `'ESTUDANTE'`). Optou-se por **não converter a caixa de `profissao` na ETL-14**, deixando essa padronização para a **ETL-17 (Padronização de profissão)**, pois a ETL-14 trata exclusivamente da integridade de encoding/acentos, apóstrofos perdidos e espaços excedentes — preservando as grafias originais para a normalização léxica e o agrupamento em `grupo_profissao` na ETL-17. Apenas espaços duplos internos (como `'TECNICO EM  MECANICA PRODUCAO'`) e eventuais espaços nas pontas são removidos aqui.

In [21]:
MAPA_TEXTO: dict[str, dict[str, str]] = {
    "municipio": {
        "SANTA BARBARA D OESTE": "SANTA BARBARA D'OESTE",
        "PALMEIRA D OESTE": "PALMEIRA D'OESTE",
        "ESTRELA D OESTE": "ESTRELA D'OESTE",
        "APARECIDA D OESTE": "APARECIDA D'OESTE",
        "SANTA RITA D OESTE": "SANTA RITA D'OESTE",
        "SAO JOAO DO PAU D ALHO": "SAO JOAO DO PAU D'ALHO",
        "SANTA CLARA D OESTE": "SANTA CLARA D'OESTE",
        "GUARANI D OESTE": "GUARANI D'OESTE",
    },
    # Mapeamento preventivo para acentos eventualmente corrompidos em amostras/extrações
    "regiao_administrativa": {
        "METROPOLITANA DE S O PAULO": "METROPOLITANA DE SÃO PAULO",
        "METROPOLITANA DE SAO PAULO": "METROPOLITANA DE SÃO PAULO",
        "S O JOSÉ DOS CAMPOS": "SÃO JOSÉ DOS CAMPOS",
        "S O JOS DOS CAMPOS": "SÃO JOSÉ DOS CAMPOS",
        "S O JOSÉ DO RIO PRETO": "SÃO JOSÉ DO RIO PRETO",
        "S O JOS DO RIO PRETO": "SÃO JOSÉ DO RIO PRETO",
        "RIBEIR O PRETO": "RIBEIRÃO PRETO",
        "MAR LIA": "MARÍLIA",
        "ARA ATUBA": "ARAÇATUBA",
    },
    "grau_de_instrucao": {
        "M DIO": "MÉDIO",
        "B SICO": "BÁSICO",
    },
}

def normalizar_espacos(df: pd.DataFrame, colunas: list[str] | None = None) -> pd.DataFrame:
    """Remove espaços nas pontas e colapsa espaços internos múltiplos nas colunas de texto."""
    df = df.copy()
    alvo = colunas if colunas is not None else listar_colunas_texto(df)
    for col in alvo:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    return df

def corrigir_encoding(
    df: pd.DataFrame,
    mapa: dict[str, dict[str, str]] | None = None,
) -> pd.DataFrame:
    """Aplica o mapa explícito {valor_errado: valor_certo} nas colunas de texto informadas."""
    df = df.copy()
    regras = mapa if mapa is not None else MAPA_TEXTO
    for col, substituicoes in regras.items():
        if col in df.columns and substituicoes:
            df[col] = df[col].replace(substituicoes)
    return df

df_antes = df
n_antes = len(df_antes)

df = normalizar_espacos(df_antes, colunas_texto)
df = corrigir_encoding(df, MAPA_TEXTO)

print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)})")

auditoria_colunas = []
for col in colunas_texto:
    s_antes = df_antes[col]
    s_depois = df[col]
    alteradas = int((s_antes.fillna("<NA>") != s_depois.fillna("<NA>")).sum())
    if alteradas > 0 or s_antes.nunique() != s_depois.nunique():
        auditoria_colunas.append({
            "coluna": col,
            "celulas_alteradas": alteradas,
            "distintos_antes": int(s_antes.nunique()),
            "distintos_depois": int(s_depois.nunique()),
        })

print("\nResumo das alterações por coluna:")
display(pd.DataFrame(auditoria_colunas))

auditoria_mapa = []
for col, regras in MAPA_TEXTO.items():
    if col not in df_antes.columns:
        continue
    for errado, certo in regras.items():
        qtd = int((df_antes[col] == errado).sum())
        if qtd > 0:
            auditoria_mapa.append({
                "coluna": col,
                "valor_errado": errado,
                "valor_certo": certo,
                "linhas_corrigidas": qtd,
            })

print("\nOcorrências efetivamente corrigidas pelo MAPA_TEXTO:")
display(pd.DataFrame(auditoria_mapa))

linhas: 806977 -> 806977 (removidas: 0)



Resumo das alterações por coluna:


,coluna,celulas_alteradas,distintos_antes,distintos_depois
0,municipio,4665,645,645
1,profissao,1,814,814
2,nacionalidade,711291,90,90



Ocorrências efetivamente corrigidas pelo MAPA_TEXTO:


,coluna,valor_errado,valor_certo,linhas_corrigidas
0,municipio,SANTA BARBARA D OESTE,SANTA BARBARA D'OESTE,4111
1,municipio,PALMEIRA D OESTE,PALMEIRA D'OESTE,231
2,municipio,ESTRELA D OESTE,ESTRELA D'OESTE,161
3,municipio,APARECIDA D OESTE,APARECIDA D'OESTE,83
4,municipio,SANTA RITA D OESTE,SANTA RITA D'OESTE,36
5,municipio,SAO JOAO DO PAU D ALHO,SAO JOAO DO PAU D'ALHO,18
6,municipio,SANTA CLARA D OESTE,SANTA CLARA D'OESTE,15
7,municipio,GUARANI D OESTE,GUARANI D'OESTE,10


### 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [22]:
depois = diagnostico(df)
comparacao = antes.join(depois, suffixes=("_antes", "_depois"))
comparacao

,dtype_antes,pct_nulo_antes,n_distintos_antes,dtype_depois,pct_nulo_depois,n_distintos_depois
id_sinistro,int64,0.00,402695,int64,0.00,402695
id_pessoa,int64,0.00,806977,int64,0.00,806977
id_veiculo,Int64,21.04,554294,Int64,21.04,554294
cod_ibge,int64,0.00,645,int64,0.00,645
municipio,str,0.00,645,str,0.00,645
regiao_administrativa,str,0.00,16,str,0.00,16
tipo_via,str,0.00,3,str,0.00,3
tipo_veiculo_vitima,str,20.98,7,str,20.98,7
tipo_de_vitima,str,0.00,4,str,0.00,4
modo_transporte_vitima,str,20.54,8,str,20.54,8


In [23]:
det_depois, res_depois = _p4_texto(df[colunas_texto])
print("Problemas de texto restantes (_p4_texto):")
display(res_depois)
display(det_depois)

# Conferência da integridade 1:1 de cod_ibge -> municipio (pré-requisito da ETL-21)
mun_por_ibge = df.groupby("cod_ibge")["municipio"].nunique()
inconsistentes_ibge = mun_por_ibge[mun_por_ibge > 1]
print(f"Códigos IBGE distintos: {len(mun_por_ibge)} | Municípios distintos: {df['municipio'].nunique()}")
print(f"Códigos IBGE com mais de um município: {len(inconsistentes_ibge)}")
assert len(inconsistentes_ibge) == 0, "Há cod_ibge associado a mais de um município!"
assert len(df) == n_antes, "Nenhuma linha deve ser removida na ETL-14!"
assert (df_antes.isna().sum() == df.isna().sum()).all(), "Contagem de nulos alterada indevidamente!"

Problemas de texto restantes (_p4_texto):


,coluna,motivo,valores_distintos,linhas
0,profissao,letra_isolada,1,1


,coluna,valor,n,motivo
0,profissao,Tecnico em raio x,1,letra_isolada


Códigos IBGE distintos: 645 | Municípios distintos: 645
Códigos IBGE com mais de um município: 0


### 6. Salvar a saída

In [24]:
df.to_parquet(SAIDA, index=False)
print("salvo em:", SAIDA, "| shape:", df.shape)

salvo em: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\03_texto.parquet | shape: (806977, 33)


### 7. Resumo

**O que foi feito:** Limpeza de texto, verificação de encoding e padronização ortográfica de municípios em todas as 15 colunas textuais da base unificada (`806.977` linhas × `33` colunas), mantendo `0` linhas removidas e `0` alterações na contagem de nulos.
- `normalizar_espacos()`: removeu espaços nas pontas e colapsou espaços internos múltiplos. Corrigiu **711.291** registros de `nacionalidade` (todos os 90 valores distintos vinham preenchidos com espaços à direita) e **1** registro de `profissao` com espaço duplo (`'TECNICO EM  MECANICA PRODUCAO'` → `'TECNICO EM MECANICA PRODUCAO'`).
- `corrigir_encoding()`: aplicou o `MAPA_TEXTO` (`{valor_errado: valor_certo}`) e restaurou o apóstrofo em **8 municípios** (`4.665` linhas no total: `SANTA BARBARA D'OESTE`, `PALMEIRA D'OESTE`, `ESTRELA D'OESTE`, `APARECIDA D'OESTE`, `SANTA RITA D'OESTE`, `SAO JOAO DO PAU D'ALHO`, `SANTA CLARA D'OESTE`, `GUARANI D'OESTE`), preservando a relação 1:1 com os 645 códigos IBGE. Também manteve entradas preventivas no mapa para acentos que apareceram corrompidos apenas em amostras antigas (`METROPOLITANA DE S O PAULO`, `M DIO`), confirmando `0` ocorrências corrompidas na leitura em `latin-1`.

**Por quê:** Espaços excedentes e grafias divergentes quebram agrupamentos, `JOIN`s e a criação das dimensões `dim_localidade` (`ETL-21`) e `dim_profissao` (`ETL-17` / `ETL-22`). A conversão de caixa de `profissao` (266 valores em caixa mista/minúscula) foi intencionalmente deixada para a `ETL-17`, onde será tratada junto com a unificação léxica e o agrupamento em `grupo_profissao`.

**Resultado:** **715.957** células corrigidas sem perda de registros; no diagnóstico final com `_p4_texto`, resta apenas 1 falso positivo conhecido (`'Tecnico em raio x'`, 1 linha, sinalizado por conter a letra `'x'`).

---

## Etapa 04 · Datas de sinistro e óbito (ETL-15)

- **Etapa:** Conversão e validação de datas
- **Responsável:** Samuel (@safn007)
- **Issue:** #17
- **Entrada:** `data/intermediarios/03_texto.parquet`
- **Saída:** `intermediarios/04_datas.parquet`

### 1. Setup

In [25]:
ENTRADA = INTERM / "03_texto.parquet"
SAIDA = INTERM / "04_datas.parquet"

### 2. Carregar a entrada

In [26]:
df = pd.read_parquet(ENTRADA)

print("shape:", df.shape)
print("colunas:", df.columns.tolist())
df.head()

shape: (806977, 33)
colunas: ['id_sinistro', 'id_pessoa', 'id_veiculo', 'cod_ibge', 'municipio', 'regiao_administrativa', 'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima', 'sexo', 'idade', 'gravidade_lesao', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'profissao', 'grau_de_instrucao', 'nacionalidade', 'data_sinistro', 'ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'ano_mes_sinistro', 'data_obito', 'ano_obito', 'mes_obito', 'dia_obito', 'ano_mes_obito', 'local_obito', 'local_via', 'tempo_sinistro_obito', 'ano_origem', 'arquivo_origem']


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,<NA>,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,<NA>,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


### 3. Diagnóstico ANTES

In [27]:
antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


### 4. Transformações

#### 4A. - Configuração e conversão

In [28]:
# As chaves são as datas completas. As tuplas seguem a ordem
# (coluna de ano, coluna de mês, coluna de dia, coluna de ano/mês).
CAMPOS = {
    "data_sinistro": ("ano_sinistro", "mes_sinistro", "dia_sinistro", "ano_mes_sinistro"),
    "data_obito": ("ano_obito", "mes_obito", "dia_obito", "ano_mes_obito"),
}
DATA_SINISTRO, DATA_OBITO = CAMPOS
DERIVADAS = [coluna for grupo in CAMPOS.values() for coluna in grupo]

if len(set(DERIVADAS)) != 8 or set(CAMPOS) & set(DERIVADAS):
    raise ValueError("Confira se há oito derivadas distintas das duas datas completas.")

faltantes = set(CAMPOS).union(DERIVADAS) - set(df.columns)
if faltantes:
    raise KeyError(f"Confira os nomes no resultado da seção 2. Faltam: {sorted(faltantes)}")

def limpar_texto(serie: pd.Series) -> pd.Series:
    """Remove espaços e considera campos vazios como ausentes."""
    return serie.astype("string").str.strip().replace("", pd.NA)

def converter_datas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Converte datas e distingue valores ausentes de datas inválidas."""
    resultado = df.copy()
    auditoria = []

    for coluna in CAMPOS:
        original = df[coluna]
        texto = limpar_texto(original)

        if pd.api.types.is_datetime64_any_dtype(original.dtype):
            data = original.dt.normalize()
        else:
            data = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
            iso = pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
            data = data.fillna(iso)

        auditoria.append({
            "coluna": coluna,
            "ausentes": int(texto.isna().sum()),
            "preenchidas_inválidas": int((texto.notna() & data.isna()).sum()),
        })
        resultado[coluna] = data

    return resultado, pd.DataFrame(auditoria).set_index("coluna")

#### 4B. - Conferência e cálculo

In [29]:
def normalizar_ano_mes(texto: pd.Series) -> pd.Series:
    """Transforma AAAA/MM ou MM/AAAA em AAAAMM para comparação."""
    ano_primeiro = texto.str.extract(r"^(\d{4})[/-](\d{1,2})$")
    mes_primeiro = texto.str.extract(r"^(\d{1,2})[/-](\d{4})$")

    ano = pd.to_numeric(ano_primeiro[0].fillna(mes_primeiro[1]), errors="coerce")
    mes = pd.to_numeric(ano_primeiro[1].fillna(mes_primeiro[0]), errors="coerce")
    return (ano * 100 + mes).where(mes.between(1, 12))

def conferir_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    """Compara as derivadas originais com as datas já convertidas."""
    auditoria = []

    for coluna_data, derivadas in CAMPOS.items():
        data = df[coluna_data]
        esperados = (
            data.dt.year,
            data.dt.month,
            data.dt.day,
            data.dt.year * 100 + data.dt.month,
        )
        limites = ((1, 9999), (1, 12), (1, 31), None)

        for coluna, esperado, limite in zip(derivadas, esperados, limites):
            texto = limpar_texto(df[coluna])

            if limite is None:
                informado = normalizar_ano_mes(texto)
            else:
                informado = pd.to_numeric(texto, errors="coerce")
                valido = informado.between(*limite) & informado.mod(1).eq(0)
                informado = informado.where(valido)

            comparavel = data.notna() & informado.notna()
            divergente = comparavel & informado.ne(esperado)

            auditoria.append({
                "coluna": coluna,
                "comparáveis": int(comparavel.sum()),
                "divergências": int(divergente.sum()),
                "derivada_ausente_com_data_válida": int((texto.isna() & data.notna()).sum()),
                "derivada_preenchida_inválida": int((texto.notna() & informado.isna()).sum()),
                "derivada_preenchida_sem_data_válida": int((texto.notna() & data.isna()).sum()),
            })

    return pd.DataFrame(auditoria).set_index("coluna")

def calcular_dias_ate_obito(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula a diferença em dias corridos e preserva os intervalos não calculáveis."""
    resultado = df.copy()
    intervalo = resultado[DATA_OBITO] - resultado[DATA_SINISTRO]
    resultado["dias_ate_obito"] = intervalo.dt.days.astype("Int64")
    return resultado

def remover_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove somente as oito colunas derivadas já conferidas."""
    return df.drop(columns=DERIVADAS)

#### 4C. - Execução e evidências

In [30]:
n_antes = len(df)
colunas_antes = set(df.columns)

df, auditoria_datas = converter_datas(df)
auditoria_derivadas = conferir_derivadas(df)
df = calcular_dias_ate_obito(df)

display(auditoria_datas)
display(auditoria_derivadas)

dias = df["dias_ate_obito"]
obito_sem_sinistro = df[DATA_OBITO].notna() & df[DATA_SINISTRO].isna()
sinistro_fora_do_periodo = (
    df[DATA_SINISTRO].notna() & ~df[DATA_SINISTRO].dt.year.isin([2021, 2022, 2023])
)

regras = pd.Series({
    "sinistros fora de 2021–2023": int(sinistro_fora_do_periodo.sum()),
    "óbitos com data, mas sem data de sinistro válida": int(obito_sem_sinistro.sum()),
    "óbitos anteriores ao sinistro": int(dias.lt(0).sum()),
    "óbitos no mesmo dia": int(dias.eq(0).sum()),
    "intervalos não calculáveis": int(dias.isna().sum()),
}, name="linhas")

display(regras.to_frame())

df = remover_derivadas(df)

assert len(df) == n_antes
assert colunas_antes - set(df.columns) == set(DERIVADAS)
assert {DATA_SINISTRO, DATA_OBITO, "dias_ate_obito"} <= set(df.columns)

print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)})")
print("colunas removidas:", DERIVADAS)

,ausentes,preenchidas_inválidas
coluna,,
data_sinistro,0,0
data_obito,791184,0


,comparáveis,divergências,derivada_ausente_com_data_válida,derivada_preenchida_inválida,derivada_preenchida_sem_data_válida
coluna,,,,,
ano_sinistro,806977,0,0,0,0
mes_sinistro,806977,0,0,0,0
dia_sinistro,806977,0,0,0,0
ano_mes_sinistro,806977,0,0,0,0
ano_obito,15793,0,0,0,0
mes_obito,15793,0,0,0,0
dia_obito,15793,0,0,0,0
ano_mes_obito,15793,0,0,0,0


,linhas
sinistros fora de 2021–2023,0
"óbitos com data, mas sem data de sinistro válida",0
óbitos anteriores ao sinistro,0
óbitos no mesmo dia,12805
intervalos não calculáveis,791184


linhas: 806977 -> 806977 (removidas: 0)
colunas removidas: ['ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'ano_mes_sinistro', 'ano_obito', 'mes_obito', 'dia_obito', 'ano_mes_obito']


### 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [31]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


### 6. Salvar a saída

In [32]:
df.to_parquet(SAIDA, index=False)
print("salvo em:", SAIDA, "| shape:", df.shape)

salvo em: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\04_datas.parquet | shape: (806977, 26)


### 7. Resumo

A ETL-15 converteu as datas de sinistro e óbito e conferiu as oito colunas derivadas de ano, mês, dia e ano/mês. As **806.977 datas de sinistro** eram válidas e estavam dentro do período de 2021 a 2023. Entre os **15.793 registros com data de óbito preenchida**, não houve datas inválidas nem óbitos anteriores ao sinistro.

Não foram encontradas divergências entre as datas completas e suas derivadas nas linhas comparáveis. Após a conferência, as oito colunas derivadas foram removidas por serem redundantes. Nenhuma linha foi excluída.

A coluna `dias_ate_obito` foi calculada para **15.793 registros**. Em **12.805**, o óbito ocorreu no mesmo dia do sinistro e o intervalo foi zero. Nos outros **2.988**, o óbito ocorreu em data posterior. Para os **791.184 registros sem data de óbito**, o intervalo permaneceu nulo.

---

## Etapa 05 · Nulos e categorias (ETL-16)

- **Etapa:** ETL-16, Nulos e categorias
- **Responsável:** Pedro Wellington
- **Issue:** ETL-16
- **Entrada:** `data/intermediarios/04_datas.parquet`
- **Saída:** `data/intermediarios/05_nulos_categorias.parquet`

### 1. Setup

In [33]:
ENTRADA = INTERM / "04_datas.parquet"             # saída da etapa anterior (ETL-15)
SAIDA   = INTERM / "05_nulos_categorias.parquet"  # resultado desta etapa

### 2. Carregar a entrada

In [34]:
df = pd.read_parquet(ENTRADA)
print('entrada:', ENTRADA.name, '| shape:', df.shape)
df.head()

entrada: 04_datas.parquet | shape: (806977, 26)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,grau_de_instrucao,nacionalidade,data_sinistro,data_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem,dias_ate_obito
0,2240338,1979797,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,MÉDIO,BRASILEIRA,2021-01-01,NaT,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv,<NA>
1,2240338,1979798,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,BRASILEIRA,2021-01-01,NaT,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv,<NA>
2,1447800,1284984,<NA>,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,SUPERIOR,BRASILEIRA,2021-01-01,NaT,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv,<NA>
3,1783410,563279,2648403,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,BRASILEIRA,2021-01-01,NaT,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv,<NA>
4,1798856,687322,<NA>,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,SUPERIOR,BRASILEIRA,2021-01-01,NaT,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv,<NA>


### 3. Diagnóstico ANTES

In [35]:
antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


#### 4.1 Marcar quem morreu
Cria a coluna `flag_obito`: True se `data_obito` está preenchida. Resultado esperado: 15.793 óbitos.

In [36]:
def criar_flag_obito(df: pd.DataFrame) -> pd.DataFrame:
    """Cria flag_obito: True se data_obito está preenchida (a pessoa morreu)."""
    df = df.copy()
    df['flag_obito'] = df['data_obito'].notna()
    return df

df = criar_flag_obito(df)
print(df['flag_obito'].value_counts())

flag_obito
False    791184
True      15793
Name: count, dtype: int64


#### 4.2 Testar a hipótese dos pedestres
Entre as linhas sem veículo, quantas são de cada tipo de vítima, por ano. Só consulta, não altera nada.

In [37]:
# linhas sem veículo
sem_veiculo = df[df['tipo_veiculo_vitima'].isna()]

# quantas de cada tipo de vítima, por ano
print(pd.crosstab(sem_veiculo['ano_origem'], sem_veiculo['tipo_de_vitima'], margins=True))

# a mesma tabela, em % de cada ano
print((pd.crosstab(sem_veiculo['ano_origem'], sem_veiculo['tipo_de_vitima'], normalize='index') * 100).round(2))

tipo_de_vitima  CONDUTOR  NAO DISPONIVEL  PASSAGEIRO  PEDESTRE     All
ano_origem                                                            
2021                1134           32779        6811     14468   55192
2022                1324           28653        7108     16181   53266
2023                1164           37072        7083     15524   60843
All                 3622           98504       21002     46173  169301
tipo_de_vitima  CONDUTOR  NAO DISPONIVEL  PASSAGEIRO  PEDESTRE
ano_origem                                                    
2021                2.05           59.39       12.34     26.21
2022                2.49           53.79       13.34     30.38
2023                1.91           60.93       11.64     25.51


#### 4.3 Nulos de veículo
Pedestre sem veículo vira `NÃO SE APLICA`. Condutor, passageiro e tipo desconhecido sem veículo viram `NÃO INFORMADO`.

In [38]:
def tratar_nulos_veiculo(df: pd.DataFrame) -> pd.DataFrame:
    """Veículo nulo: pedestre vira NÃO SE APLICA; os demais viram NÃO INFORMADO."""
    df = df.copy()
    pedestre = df['tipo_de_vitima'] == 'PEDESTRE'
    for col in ['tipo_veiculo_vitima', 'modo_transporte_vitima']:
        nulo = df[col].isna()
        df.loc[nulo & pedestre, col] = 'NÃO SE APLICA'
        df.loc[nulo & ~pedestre, col] = 'NÃO INFORMADO'
    return df

df = tratar_nulos_veiculo(df)
print(df['tipo_veiculo_vitima'].value_counts(dropna=False))

tipo_veiculo_vitima
AUTOMOVEL         315274
MOTOCICLETA       274658
NÃO INFORMADO     123128
NÃO SE APLICA      46173
CAMINHAO           22332
ONIBUS             16336
OUTROS              7472
BICICLETA           1124
NAO DISPONIVEL       480
Name: count, dtype: int64


#### 4.4 Conferir as idades
Verifica mínimo, máximo e valores fora de 0 a 110. Só consulta: nenhuma idade inválida foi encontrada.

In [39]:
# resumo das idades, por ano
print(df.groupby('ano_origem')['idade'].describe())

# quantas idades fora do intervalo plausível (0 a 110)
invalidas = df[(df['idade'] < 0) | (df['idade'] > 110)]
print(len(invalidas), 'idades fora de 0 a 110')
print(invalidas.groupby('ano_origem').size())

               count       mean        std  min   25%   50%   75%    max
ano_origem                                                              
2021        257411.0  36.713528  14.778949  0.0  25.0  34.0  46.0  102.0
2022        262331.0  37.416268  15.113480  0.0  25.0  35.0  47.0   99.0
2023        278846.0  37.671173  15.292993  0.0  25.0  36.0  48.0  100.0
0 idades fora de 0 a 110
Series([], dtype: int64)


#### 4.5 Nulos de texto
`profissao`, `nacionalidade` e `local_via` vazios viram `NÃO INFORMADO`. `local_obito` vazio vira `NÃO SE APLICA` para quem não morreu.

In [40]:
def tratar_nulos_texto(df: pd.DataFrame) -> pd.DataFrame:
    """local_obito vazio de quem não morreu = NÃO SE APLICA; os demais nulos de texto = NÃO INFORMADO."""
    df = df.copy()

    # quem não morreu não tem local de óbito
    nulo = df['local_obito'].isna()
    df.loc[nulo & ~df['flag_obito'], 'local_obito'] = 'NÃO SE APLICA'
    df.loc[nulo & df['flag_obito'], 'local_obito'] = 'NÃO INFORMADO'

    # campos que deviam ter valor e vieram vazios
    for col in ['profissao', 'nacionalidade', 'local_via']:
        df[col] = df[col].fillna('NÃO INFORMADO')
    return df

df = tratar_nulos_texto(df)

# o que ainda tem nulo
nulos = df.isna().sum()
print(nulos[nulos > 0])

id_veiculo              169754
idade                     8389
grau_de_instrucao       632289
data_obito              791184
tempo_sinistro_obito    791184
dias_ate_obito          791184
dtype: int64


#### 4.6 Números sem casa decimal
Troca `float` por `Int64` (inteiro que aceita nulo) em idade, `id_veiculo` e partes da data de óbito.

In [41]:
def converter_inteiros(df: pd.DataFrame) -> pd.DataFrame:
    """Troca float por Int64 (inteiro que aceita nulo): idade, id_veiculo e partes da data de óbito."""
    df = df.copy()
    colunas_alvo = [c for c in ['idade', 'id_veiculo', 'ano_obito', 'mes_obito', 'dia_obito'] if c in df.columns]
    for col in colunas_alvo:
        df[col] = df[col].astype('Int64')
    return df

df = converter_inteiros(df)
colunas_int = [c for c in ['idade', 'id_veiculo', 'ano_obito', 'mes_obito', 'dia_obito'] if c in df.columns]
print(df[colunas_int].dtypes)
df[['idade', 'id_veiculo']].head()

idade         Int64
id_veiculo    Int64
dtype: object


,idade,id_veiculo
0,31,2874666
1,34,2874666
2,65,<NA>
3,26,2648403
4,66,<NA>


#### 4.7 Grau de instrução
Primeiro vê o % preenchido mês a mês (sempre ~20% a 23%, não existe período confiável). Depois os nulos viram `NÃO INFORMADO`.

In [42]:
# % de grau_de_instrucao preenchido, mês a mês (36 meses)
competencia = df['ano_mes_sinistro'] if 'ano_mes_sinistro' in df.columns else df['data_sinistro'].dt.strftime('%Y-%m')
tabela = df.groupby(competencia)['grau_de_instrucao'].apply(lambda s: round(s.notna().mean() * 100, 2))
print(tabela.to_string())

data_sinistro
2021-01    22.62
2021-02    21.58
2021-03    20.99
2021-04    21.32
2021-05    21.62
2021-06    20.90
2021-07    20.92
2021-08    21.61
2021-09    22.07
2021-10    22.19
2021-11    22.10
2021-12    22.53
2022-01    22.75
2022-02    21.98
2022-03    21.98
2022-04    22.75
2022-05    21.85
2022-06    21.97
2022-07    22.81
2022-08    21.52
2022-09    21.41
2022-10    22.22
2022-11    21.54
2022-12    23.59
2023-01    22.63
2023-02    21.37
2023-03    21.70
2023-04    22.74
2023-05    20.48
2023-06    20.32
2023-07    20.69
2023-08    20.53
2023-09    20.38
2023-10    19.83
2023-11    21.39
2023-12    21.61


In [43]:
def tratar_grau_instrucao(df: pd.DataFrame) -> pd.DataFrame:
    """grau_de_instrucao vazio vira NÃO INFORMADO (a coluna é ~78% vazia em todos os meses)."""
    df = df.copy()
    df['grau_de_instrucao'] = df['grau_de_instrucao'].fillna('NÃO INFORMADO')
    return df

df = tratar_grau_instrucao(df)
print(df['grau_de_instrucao'].value_counts(dropna=False))

grau_de_instrucao
NÃO INFORMADO    632289
SUPERIOR         105448
MÉDIO             45767
BÁSICO            23473
Name: count, dtype: int64


#### 4.8 "NAO DISPONIVEL" em outras colunas (só consulta)

In [44]:
# colunas de texto da tabela
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# em cada uma, quantas linhas têm exatamente "NAO DISPONIVEL"
contagem = {c: int((df[c] == 'NAO DISPONIVEL').sum()) for c in colunas_texto}

# mostra só as colunas onde o texto aparece
{c: n for c, n in contagem.items() if n > 0}

{'tipo_via': 973,
 'tipo_veiculo_vitima': 480,
 'tipo_de_vitima': 112828,
 'modo_transporte_vitima': 480,
 'sexo': 4366,
 'gravidade_lesao': 70314,
 'faixa_etaria_demografica': 8389,
 'faixa_etaria_legal': 8389,
 'local_obito': 105,
 'local_via': 117298}

#### 4.9 "NAO DISPONIVEL" vira NÃO INFORMADO

In [45]:
COLUNAS_NAO_DISP = [
    'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima',
    'sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'local_obito', 'local_via',
]

def tratar_nao_disponivel(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """'NAO DISPONIVEL' é nulo disfarçado: vira 'NÃO INFORMADO', igual aos outros nulos."""
    df = df.copy()
    for c in colunas:
        df[c] = df[c].replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    return df

df = tratar_nao_disponivel(df, COLUNAS_NAO_DISP)

# conferência: não deve sobrar nenhum
print({c: int((df[c] == 'NAO DISPONIVEL').sum()) for c in COLUNAS_NAO_DISP})

{'tipo_via': 0, 'tipo_veiculo_vitima': 0, 'tipo_de_vitima': 0, 'modo_transporte_vitima': 0, 'sexo': 0, 'faixa_etaria_demografica': 0, 'faixa_etaria_legal': 0, 'local_obito': 0, 'local_via': 0}


### 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [46]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,0.00,8
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,0.00,9


### 6. Salvar a saída

In [47]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\05_nulos_categorias.parquet | shape: (806977, 27)


### 7. Resumo

**Nulos disfarçados:** o texto "NAO DISPONIVEL" aparecia em 9 colunas (`tipo_via`, `tipo_veiculo_vitima`, `tipo_de_vitima`, `modo_transporte_vitima`, `sexo`, `faixa_etaria_demografica`, `faixa_etaria_legal`, `local_obito` e `local_via`) e passou a ser `NÃO INFORMADO`, igual aos outros nulos. `gravidade_lesao` ficou como está, para a ETL-19.

---

## Etapa 06 · Padronização de profissão (ETL-17)

- **Etapa:** ETL-17, Padronização de profissão
- **Responsável:** Pedro Wellington
- **Issue:** ETL-17
- **Entrada:** `data/intermediarios/05_nulos_categorias.parquet`
- **Saída:** `data/intermediarios/06_limpo.parquet`

### 1. Setup

In [48]:
ENTRADA = INTERM / "05_nulos_categorias.parquet"  # saída da ETL-16
SAIDA   = INTERM / "06_limpo.parquet"          # resultado desta etapa

### 2. Carregar a entrada

In [49]:
df = pd.read_parquet(ENTRADA)
print('shape:', df.shape)

# quantas grafias distintas de profissão e as 30 mais comuns
print('grafias distintas:', df['profissao'].nunique())
df['profissao'].value_counts().head(30)

shape: (806977, 27)
grafias distintas: 814


profissao
NÃO INFORMADO                  86198
MOTORISTA                      70309
AUTONOMO(A)                    46750
MOTO-BOY                       43734
APOSENTADO(A)                  36703
ESTUDANTE                      31045
AJUDANTE                       24524
OUTRA                          21544
DO LAR                         17341
VENDEDOR(A)                    17016
DESEMPREGADO(A)                15476
EMPRESARIO(A)                  13890
PROFESSOR(A)                   12956
PEDREIRO                       12955
COMERCIANTE                    12292
MECANICO(A)                    10714
ENTREGADOR(A)                  10465
OPERADOR DE MAQUINA             8860
AUXILIAR ADMINISTRATIVO         8354
AUXILIAR DE SERVIÇOS GERAIS     7701
AUXILIAR GERAL                  7195
VIGILANTE                       7154
POLICIAL MILITAR                7061
ATENDENTE                       6723
ENGENHEIRO(A)                   6681
AUXILIAR DE PRODUCAO            6653
ADMINISTRADOR(A)            

### 3. Diagnóstico ANTES

In [50]:
antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,0.00,8
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,0.00,9


### 4. Transformações

Cada tratamento está numa função. Nenhum deles remove linhas.

#### 4.1 Padronizar a escrita

In [51]:
def padronizar_profissao(df: pd.DataFrame) -> pd.DataFrame:
    """Deixa a escrita igual: maiúsculas, sem acento, sem (A), sem hífen e sem espaço duplo."""
    df = df.copy()
    s = df['profissao'].str.upper()

    # tira acentos (Á vira A)
    s = s.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')

    s = s.str.replace(r'\(A\)|\(O\)', '', regex=True)   # MECANICO(A) vira MECANICO
    s = s.str.replace(r'[-/]', ' ', regex=True)         # MOTO-BOY vira MOTO BOY
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()  # espaço duplo e pontas

    df['profissao'] = s
    return df

n_antes = len(df); distintas_antes = df['profissao'].nunique()
df = padronizar_profissao(df)
print('linhas:', n_antes, '->', len(df))
print('grafias distintas:', distintas_antes, '->', df['profissao'].nunique())

linhas: 806977 -> 806977
grafias distintas: 814 -> 562


#### 4.2 Ver o que ainda sobra (só consulta)

In [52]:
# as 80 profissões mais comuns depois de padronizar (cobrem quase toda a base)
contagem = df['profissao'].value_counts()
print('cobertura das 80 maiores:', round(contagem.head(80).sum() / contagem.sum() * 100, 1), '%')
print(' | '.join(f'{p} ({n})' for p, n in contagem.head(80).items()))

cobertura das 80 maiores: 88.2 %
NAO INFORMADO (86198) | MOTORISTA (70535) | AUTONOMO (46912) | MOTO BOY (43846) | APOSENTADO (37019) | ESTUDANTE (31220) | AJUDANTE (24595) | OUTRA (21544) | DO LAR (17341) | VENDEDOR (17065) | DESEMPREGADO (15738) | EMPRESARIO (13925) | PEDREIRO (13020) | PROFESSOR (12977) | COMERCIANTE (12333) | MECANICO (10771) | ENTREGADOR (10485) | OPERADOR DE MAQUINA (8874) | AUXILIAR ADMINISTRATIVO (8372) | AUXILIAR DE SERVICOS GERAIS (7746) | AUXILIAR GERAL (7202) | VIGILANTE (7184) | POLICIAL MILITAR (7090) | ATENDENTE (6746) | ENGENHEIRO (6690) | AUXILIAR DE PRODUCAO (6677) | ADMINISTRADOR (6140) | GERENTE (6139) | ELETRICISTA (6036) | TECNICO (5978) | ENFERMEIRO (5852) | PINTOR (5021) | ANALISTA (4818) | OPERADOR (4731) | ADVOGADO (4603) | FUNCION.PUBLICO MUNICIPAL (4014) | ASSISTENTE ADMINISTRATIVO (3954) | DOMESTICA (3877) | PORTEIRO (3843) | AUXILIAR DE ENFERMAGEM (3810) | METALURGICO (3451) | COZINHEIRO (3383) | ACOUGUEIRO (3373) | BALCONISTA (3182) | SEG

#### 4.3 Unificar grafias que ainda diferem

In [53]:
# grafias que significam a mesma coisa e a padronização de escrita não juntou
UNIFICAR = {
    'MOTOBOY': 'MOTO BOY',
    'MOTOFRETISTA': 'MOTO FRETISTA',
}

df['profissao'] = df['profissao'].replace(UNIFICAR)
print('grafias distintas:', df['profissao'].nunique())

grafias distintas: 561


#### 4.4 Criar o grupo da profissão

In [54]:
# cada grupo tem palavras-chave; vale o primeiro grupo que combinar (a ordem importa)
REGRAS = [
    ('SEM OCUPACAO', r'ESTUDANTE|APOSENTAD|DESEMPREGAD|DESOCUPAD|DO LAR|PENSIONISTA'),
    ('SAUDE E EDUCACAO', r'ENFERM|MEDIC|FISIOTERAP|DENTISTA|PSICOLOG|FARMAC|NUTRICION|VETERINAR|PROFESSOR|PEDAGOG|INSTRUTOR'),
    ('TRANSPORTE E ENTREGA', r'MOTORISTA|MOTO ?BOY|MOTO ?FRETISTA|MOTOTAXISTA|MOTOCICLISTA|ENTREGADOR|TAXISTA|CAMINHONEIRO|CONDUTOR|FRETISTA|COBRADOR|MANOBRISTA'),
    ('SEGURANCA E SERVICO PUBLICO', r'VIGILANTE|POLICIAL|SEGURANCA|PORTEIRO|BOMBEIRO|MILITAR|FUNCION|SERVIDOR PUBLICO|GUARDA'),
    ('TECNICOS E NIVEL SUPERIOR', r'ENGENHEIR|ADVOGAD|ARQUITET|ANALISTA DE SISTEMAS|PROGRAMADOR|TECNICO|DESIGNER|JORNALISTA|PUBLICITARIO'),
    ('ADMINISTRATIVO', r'ADMINISTRAT|ADMINISTRADOR|ANALISTA|SUPERVISOR|GERENTE|CONTADOR|SECRETARI|ENCARREGADO|BANCARI|ASSISTENTE|COORDENADOR|DIRETOR|RECEPCIONISTA|ESTAGIARIO|AUXILIAR DE ESCRITORIO|ALMOXARIFE|ESTOQUISTA|CONFERENTE|CONSULTOR'),
    ('COMERCIO E VENDAS', r'VENDEDOR|COMERCIANTE|REPOSITOR|REPRESENTANTE|PROMOTOR|CAIXA|BALCONISTA|ATENDENTE|COMERCIAL|CORRETOR'),
    ('AGRICULTURA E CAMPO', r'AGRICULTOR|LAVRADOR|TRATORISTA|PECUARIST|JARDINEIR|CAMPO'),
    ('INDUSTRIA E CONSTRUCAO', r'PEDREIRO|PINTOR|MECANICO|OPERADOR|METALURG|SOLDADOR|SERRALHEIR|MONTADOR|ELETRICISTA|PRODUCAO|MARCENEIR|SERVENTE|AJUDANTE|CARPINTEIR|ENCANADOR|MANUTENCAO|OPERARIO|COSTUREIR|FUNILEIR|SAPATEIR|VIDRACEIR'),
    ('SERVICOS PESSOAIS E DOMESTICOS', r'AUXILIAR GERAL|SERVICOS GERAIS|DOMESTIC|DIARISTA|LIMPEZA|COZINH|CABELEIREIR|MANICURE|BARBEIR|GARCOM|GARCON|ACOUGUEIR|FRENTISTA|FAXINEIR|CUIDADOR'),
    ('AUTONOMO E EMPRESARIO', r'AUTONOMO|EMPRESARIO|PROPRIETARIO'),
]

def classificar_grupos(profissao: pd.Series) -> pd.Series:
    """Começa tudo como OUTROS e preenche os grupos de trás para frente, para valer o primeiro."""
    grupo = pd.Series('OUTROS', index=profissao.index)
    for nome, padrao in reversed(REGRAS):
        grupo[profissao.str.contains(padrao, na=False)] = nome
    return grupo

df['grupo_profissao'] = classificar_grupos(df['profissao'])
sem_prof = df['profissao'].isna() | df['profissao'].isin(['NAO INFORMADO', 'NÃO INFORMADO'])
df.loc[sem_prof, 'grupo_profissao'] = 'NÃO INFORMADO'

print(df['grupo_profissao'].value_counts())
# o que ficou em OUTROS: serve para achar palavras-chave que faltam
print(df.loc[df['grupo_profissao'] == 'OUTROS', 'profissao'].value_counts().head(25))

grupo_profissao
TRANSPORTE E ENTREGA              131023
INDUSTRIA E CONSTRUCAO            106781
SEM OCUPACAO                      102987
NÃO INFORMADO                      86198
OUTROS                             61977
AUTONOMO E EMPRESARIO              60892
ADMINISTRATIVO                     58429
COMERCIO E VENDAS                  50758
SERVICOS PESSOAIS E DOMESTICOS     44795
SAUDE E EDUCACAO                   35643
SEGURANCA E SERVICO PUBLICO        32054
TECNICOS E NIVEL SUPERIOR          28996
AGRICULTURA E CAMPO                 6444
Name: count, dtype: int64
profissao
OUTRA                   21544
INSTALADOR               1146
PADEIRO                  1040
FISCAL                   1010
ESTETICISTA               910
CONSTRUTOR                892
AGENTE PENITENCIARIO      891
INSPETOR                  890
PIZZAIOLO                 868
LAVADOR DE CARRO          843
ZELADOR                   821
BABA                      781
BORRACHEIRO               772
AGENTE DE APOIO          

### 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [55]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,0.00,8
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,0.00,9


### 6. Salvar a saída

In [56]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\06_limpo.parquet | shape: (806977, 28)


### 7. Resumo

**O que foi feito:** padronização da escrita das profissões e criação da coluna `grupo_profissao`. O resultado foi salvo em `data/intermediarios/06_limpo.parquet`.

**Padronização:** maiúsculas, sem acento, sem "(A)" e sem hífen (MOTO-BOY virou MOTO BOY). Duas grafias foram unificadas à mão (MOTOBOY e MOTOFRETISTA). As grafias distintas caíram de 814 para 561.

**Grupos:** a profissão recebe um de 11 grupos por palavras-chave, como TRANSPORTE E ENTREGA, INDUSTRIA E CONSTRUCAO e SAUDE E EDUCACAO. Profissões sem palavra-chave ficam em OUTROS (62.275 linhas, 7,7%), a maior parte da própria categoria OUTRA da fonte (21.544). Quem não tem profissão informada fica em NÃO INFORMADO (85.900 linhas).

**O que não mudou:** nenhuma linha foi removida (806.977). A coluna `profissao` continua com 10,64% de nulos, que a `dim_profissao` trata com o membro -1. A tabela final tem 34 colunas.

**Observação:** os grupos usam palavras-chave e podem ser refinados. Os grupos finais seguem a ordem das regras, e o primeiro que combina vale.

---

## Etapa 07a · dim_tempo e dim_gravidade (ETL-19)

- **Etapa:** ETL-19, dim_tempo e dim_gravidade
- **Responsável:** Davi Mello (tarefa assumida do Samuel)
- **Issue:** ETL-19
- **Entrada:** `data/intermediarios/06_limpo.parquet`
- **Saídas:** `data/intermediarios/dims/dim_tempo.parquet` e `dim_gravidade.parquet`

Cria o calendário do modelo estrela (usado duas vezes pela fato: data do sinistro e data do óbito) e a dimensão de gravidade da lesão. As duas têm os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). As colunas seguem o DDL da ETL-18 (`sql/ddl/etl_star_schema.sql`).

### 1. Setup

In [57]:
# Na execução final as duas variáveis apontam para o mesmo arquivo (a saída da etapa anterior)
ENTRADA_DATAS     = INTERM / "06_limpo.parquet"
ENTRADA_GRAVIDADE = INTERM / "06_limpo.parquet"

INICIO = pd.Timestamp("2021-01-01")   # primeiro dia do período do projeto

### 2. Carregar a entrada

In [58]:
df_datas = pd.read_parquet(ENTRADA_DATAS, columns=['id_pessoa', 'data_sinistro', 'data_obito'])
df_grav  = pd.read_parquet(ENTRADA_GRAVIDADE, columns=['id_pessoa', 'gravidade_lesao', 'flag_obito'])
print('datas:', df_datas.shape, '| gravidade:', df_grav.shape)

# os dois arquivos têm que ser da mesma base: mesmas pessoas
assert len(df_datas) == len(df_grav) and df_datas['id_pessoa'].is_unique
assert df_datas['id_pessoa'].isin(df_grav['id_pessoa']).all()
print('mesmas', len(df_datas), 'pessoas nos dois arquivos')

datas: (806977, 3) | gravidade: (806977, 3)
mesmas 806977 pessoas nos dois arquivos


In [59]:
def para_data(serie: pd.Series) -> pd.Series:
    """Aceita a data já convertida (ETL-15) ou o texto dd/mm/aaaa da fonte, e devolve só a data."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.dt.normalize()
    return pd.to_datetime(serie, format='%d/%m/%Y', errors='raise')

df_datas['data_sinistro'] = para_data(df_datas['data_sinistro'])
df_datas['data_obito'] = para_data(df_datas['data_obito'])

### 3. Diagnóstico ANTES

In [60]:
antes = pd.concat([
    diagnostico(df_datas[['data_sinistro', 'data_obito']]),
    diagnostico(df_grav[['gravidade_lesao']]),
])
antes

,dtype,pct_nulo,n_distintos
data_sinistro,datetime64[us],0.00,1095
data_obito,datetime64[us],98.04,1109
gravidade_lesao,str,0.00,5


In [61]:
# período real das datas da base
print('data_sinistro:', df_datas['data_sinistro'].min().date(), 'a', df_datas['data_sinistro'].max().date())
print('data_obito:   ', df_datas['data_obito'].min().date(), 'a', df_datas['data_obito'].max().date())
print('\ngravidade_lesao:')
print(df_grav['gravidade_lesao'].value_counts(dropna=False))

data_sinistro: 2021-01-01 a 2023-12-31
data_obito:    2021-01-01 a 2024-01-24

gravidade_lesao:
gravidade_lesao
LEVE              357297
ILESO             321999
NAO DISPONIVEL     70314
GRAVE              41574
FATAL              15793
Name: count, dtype: int64


### 4. dim_tempo

Um dia por linha, de 01/01/2021 até a maior data da base (a maior `data_obito`, porque há óbitos até 30 dias depois de sinistros de dezembro de 2023). A chave `sk_tempo` é a própria data no formato `AAAAMMDD` (ex.: `20210115`), como define o dicionário do DW.

#### 4.1 Feriados nacionais

Só feriados nacionais, como proposto no modelo estrela (ETL-10). São os 8 de data fixa mais a Sexta-feira Santa, que depende da Páscoa. Carnaval e Corpus Christi ficam de fora porque são ponto facultativo, não feriado. O Dia da Consciência Negra (20/11) só virou feriado nacional em 2024 (Lei 14.759/2023), depois do período da base, mas a regra já está na função.

In [62]:
def pascoa(ano: int) -> pd.Timestamp:
    """Domingo de Páscoa do ano (algoritmo de Meeus/Jones/Butcher, calendário gregoriano)."""
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = (h + l - 7 * m + 114) % 31 + 1
    return pd.Timestamp(ano, mes, dia)

# conferência com as datas conhecidas da Páscoa
assert [pascoa(a).date().isoformat() for a in (2021, 2022, 2023, 2024)] == \
    ['2021-04-04', '2022-04-17', '2023-04-09', '2024-03-31']

FERIADOS_FIXOS = [
    (1, 1, 'Confraternização Universal'),
    (4, 21, 'Tiradentes'),
    (5, 1, 'Dia do Trabalho'),
    (9, 7, 'Independência do Brasil'),
    (10, 12, 'Nossa Senhora Aparecida'),
    (11, 2, 'Finados'),
    (11, 15, 'Proclamação da República'),
    (12, 25, 'Natal'),
]

def feriados_nacionais(anos) -> pd.Series:
    """Série data -> nome do feriado nacional, para os anos pedidos."""
    feriados = {}
    for ano in anos:
        for mes, dia, nome in FERIADOS_FIXOS:
            feriados[pd.Timestamp(ano, mes, dia)] = nome
        feriados[pd.Timestamp(pascoa(ano).to_pydatetime() - timedelta(days=2))] = 'Sexta-feira Santa'
        if ano >= 2024:   # Lei 14.759/2023
            feriados[pd.Timestamp(ano, 11, 20)] = 'Dia Nacional de Zumbi e da Consciência Negra'
    return pd.Series(feriados, name='nome_feriado').sort_index()

#### 4.2 Criar a dimensão

In [63]:
NOMES_MES = ['JANEIRO', 'FEVEREIRO', 'MARÇO', 'ABRIL', 'MAIO', 'JUNHO', 'JULHO',
             'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']
NOMES_DIA = ['SEGUNDA-FEIRA', 'TERÇA-FEIRA', 'QUARTA-FEIRA', 'QUINTA-FEIRA',
             'SEXTA-FEIRA', 'SÁBADO', 'DOMINGO']   # 1 = segunda ... 7 = domingo

def criar_dim_tempo(inicio: pd.Timestamp, fim: pd.Timestamp) -> pd.DataFrame:
    """Um dia por linha entre inicio e fim, com chave AAAAMMDD e os membros -1 e -2."""
    datas = pd.date_range(inicio, fim, freq='D')
    feriados = feriados_nacionais(range(inicio.year, fim.year + 1))

    dim = pd.DataFrame({'data': datas})
    dim.insert(0, 'sk_tempo', (datas.year * 10000 + datas.month * 100 + datas.day).astype('int64'))
    dim['ano'] = datas.year
    dim['trimestre'] = datas.quarter
    dim['mes'] = datas.month
    dim['nome_mes'] = [NOMES_MES[m - 1] for m in datas.month]
    dim['dia'] = datas.day
    dim['ano_mes'] = datas.strftime('%Y-%m')
    dim['dia_semana'] = datas.day_of_week + 1
    dim['nome_dia_semana'] = [NOMES_DIA[d] for d in datas.day_of_week]
    dim['fim_de_semana'] = dim['dia_semana'].isin([6, 7])
    dim['nome_feriado'] = dim['data'].map(feriados)
    dim['feriado'] = dim['nome_feriado'].notna()

    # mesma ordem de colunas do DDL
    dim = dim[['sk_tempo', 'data', 'ano', 'trimestre', 'mes', 'nome_mes', 'dia', 'ano_mes',
               'dia_semana', 'nome_dia_semana', 'fim_de_semana', 'feriado', 'nome_feriado']]

    # tipos que aceitam nulo, compatíveis com o DDL (SMALLINT, BOOLEAN, VARCHAR)
    for col in ['ano', 'trimestre', 'mes', 'dia', 'dia_semana']:
        dim[col] = dim[col].astype('Int16')
    for col in ['fim_de_semana', 'feriado']:
        dim[col] = dim[col].astype('boolean')
    TEXTOS = ['nome_mes', 'ano_mes', 'nome_dia_semana', 'nome_feriado']
    for col in TEXTOS:
        dim[col] = dim[col].astype('string')

    # membros especiais, com os mesmos tipos: números e datas ficam nulos, textos recebem o rótulo
    especiais = pd.DataFrame({c: pd.Series([pd.NA, pd.NA], dtype=dim[c].dtype) for c in dim.columns[1:]})
    especiais.insert(0, 'sk_tempo', pd.Series([-1, -2], dtype='int64'))
    for col in ['nome_mes', 'ano_mes', 'nome_dia_semana']:
        especiais[col] = pd.Series(['NÃO INFORMADO', 'NÃO SE APLICA'], dtype='string')
    dim = pd.concat([especiais, dim], ignore_index=True)
    return dim

# o calendário vai até a maior data que aparece na base
FIM = max(df_datas['data_sinistro'].max(), df_datas['data_obito'].max())
assert df_datas['data_sinistro'].min() >= INICIO, 'há sinistro antes de 2021'

dim_tempo = criar_dim_tempo(INICIO, FIM)
print(len(dim_tempo), 'linhas | de', INICIO.date(), 'a', FIM.date())
dim_tempo.head(6)

1121 linhas | de 2021-01-01 a 2024-01-24


,sk_tempo,data,ano,trimestre,mes,nome_mes,dia,ano_mes,dia_semana,nome_dia_semana,fim_de_semana,feriado,nome_feriado
0,-1,NaT,<NA>,<NA>,<NA>,NÃO INFORMADO,<NA>,NÃO INFORMADO,<NA>,NÃO INFORMADO,<NA>,<NA>,<NA>
1,-2,NaT,<NA>,<NA>,<NA>,NÃO SE APLICA,<NA>,NÃO SE APLICA,<NA>,NÃO SE APLICA,<NA>,<NA>,<NA>
2,20210101,2021-01-01,2021,1,1,JANEIRO,1,2021-01,5,SEXTA-FEIRA,False,True,Confraternização Universal
3,20210102,2021-01-02,2021,1,1,JANEIRO,2,2021-01,6,SÁBADO,True,False,<NA>
4,20210103,2021-01-03,2021,1,1,JANEIRO,3,2021-01,7,DOMINGO,True,False,<NA>
5,20210104,2021-01-04,2021,1,1,JANEIRO,4,2021-01,1,SEGUNDA-FEIRA,False,False,<NA>


In [64]:
# feriados de cada ano (dias marcados na dimensão)
reais = dim_tempo[dim_tempo['sk_tempo'] > 0]
feriados_por_ano = reais[reais['feriado']].groupby('ano')['nome_feriado'].count()
print(feriados_por_ano.to_string(), '\n')
reais.loc[reais['feriado'], ['data', 'nome_dia_semana', 'nome_feriado']].head(10)

ano
2021    9
2022    9
2023    9
2024    1 



,data,nome_dia_semana,nome_feriado
2,2021-01-01,SEXTA-FEIRA,Confraternização Universal
93,2021-04-02,SEXTA-FEIRA,Sexta-feira Santa
112,2021-04-21,QUARTA-FEIRA,Tiradentes
122,2021-05-01,SÁBADO,Dia do Trabalho
251,2021-09-07,TERÇA-FEIRA,Independência do Brasil
286,2021-10-12,TERÇA-FEIRA,Nossa Senhora Aparecida
307,2021-11-02,TERÇA-FEIRA,Finados
320,2021-11-15,SEGUNDA-FEIRA,Proclamação da República
360,2021-12-25,SÁBADO,Natal
367,2022-01-01,SÁBADO,Confraternização Universal


### 5. dim_gravidade

As quatro gravidades da fonte, com `nivel_gravidade` de 0 (ileso) a 3 (fatal) para ordenar e filtrar ("grave ou pior" = nível 2 ou mais). A ETL-16 deixou `NAO DISPONIVEL` em `gravidade_lesao` para esta etapa: ele vai para o membro -1.

In [65]:
NIVEIS = {'ILESO': 0, 'LEVE': 1, 'GRAVE': 2, 'FATAL': 3}
NAO_INFORMADO = ['NAO DISPONIVEL', 'NÃO INFORMADO']

def criar_dim_gravidade(gravidade: pd.Series) -> pd.DataFrame:
    """Uma linha por gravidade, chave na ordem do nível (1 = ILESO ... 4 = FATAL), e os membros -1 e -2."""
    valores = set(gravidade.dropna().unique()) - set(NAO_INFORMADO)
    # só aceita as gravidades conhecidas, para nunca criar uma linha de lixo na dimensão
    assert valores <= set(NIVEIS), f'gravidade nova: {valores - set(NIVEIS)}'

    dim = pd.DataFrame({'gravidade_lesao': list(NIVEIS), 'nivel_gravidade': list(NIVEIS.values())})
    dim.insert(0, 'sk_gravidade', dim['nivel_gravidade'] + 1)
    especiais = pd.DataFrame({
        'sk_gravidade': [-1, -2],
        'gravidade_lesao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
        'nivel_gravidade': [pd.NA, pd.NA],
    })
    dim = pd.concat([especiais, dim], ignore_index=True)
    dim['sk_gravidade'] = dim['sk_gravidade'].astype('int64')
    dim['gravidade_lesao'] = dim['gravidade_lesao'].astype('string')
    dim['nivel_gravidade'] = dim['nivel_gravidade'].astype('Int16')
    return dim

dim_gravidade = criar_dim_gravidade(df_grav['gravidade_lesao'])
dim_gravidade

,sk_gravidade,gravidade_lesao,nivel_gravidade
0,-1,NÃO INFORMADO,<NA>
1,-2,NÃO SE APLICA,<NA>
2,1,ILESO,0
3,2,LEVE,1
4,3,GRAVE,2
5,4,FATAL,3


### 6. Conferir e salvar

As funções `atribuir_sk_tempo` e `atribuir_sk_gravidade` ligam cada pessoa à chave, e a ETL-23 (tabela fato) pode reutilizá-las. Conferências: chaves únicas, calendário sem buracos, tamanhos do DDL respeitados e todas as pessoas com chave.

In [66]:
def atribuir_sk_tempo(datas: pd.Series, sem_data: int) -> pd.Series:
    """Chave AAAAMMDD de cada data; quem não tem data recebe sem_data (-2 no óbito de quem não morreu)."""
    datas = para_data(datas)
    sk = datas.dt.year * 10000 + datas.dt.month * 100 + datas.dt.day
    return sk.fillna(sem_data).astype('int64')

def atribuir_sk_gravidade(gravidade: pd.Series) -> pd.Series:
    """Chave de cada pessoa na dim_gravidade (-1 quando NAO DISPONIVEL ou vazia)."""
    sk = gravidade.map({g: n + 1 for g, n in NIVEIS.items()})
    return sk.fillna(-1).astype('int64')

In [67]:
# 1. chaves únicas e calendário contínuo, sem buracos
reais = dim_tempo[dim_tempo['sk_tempo'] > 0]
assert dim_tempo['sk_tempo'].is_unique and reais['data'].is_unique
assert reais['sk_tempo'].is_monotonic_increasing
assert len(reais) == (FIM - INICIO).days + 1, 'faltam dias no calendário'
assert dim_gravidade['sk_gravidade'].is_unique and dim_gravidade['gravidade_lesao'].is_unique

# 2. tamanhos dos textos dentro do DDL
LIMITES = [
    (dim_tempo, {'nome_mes': 20, 'ano_mes': 15, 'nome_dia_semana': 20, 'nome_feriado': 40}),
    (dim_gravidade, {'gravidade_lesao': 20}),
]
for dim, limites in LIMITES:
    for col, limite in limites.items():
        assert dim[col].str.len().max() <= limite, f'{col} maior que {limite}'

# 3. dias reais com todos os campos preenchidos (CHECK chk_dim_tempo_campos_validos)
obrigatorias = ['data', 'ano', 'trimestre', 'mes', 'nome_mes', 'dia', 'ano_mes',
                'dia_semana', 'nome_dia_semana', 'fim_de_semana', 'feriado']
assert reais[obrigatorias].notna().all().all()

for nome, dim in [('dim_tempo', dim_tempo), ('dim_gravidade', dim_gravidade)]:
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)
    print(nome, '|', len(dim), 'linhas | salvo em dims/')

dim_tempo | 1121 linhas | salvo em dims/
dim_gravidade | 6 linhas | salvo em dims/


In [68]:
# 4. cada pessoa tem que achar uma chave
sk_data_sinistro = atribuir_sk_tempo(df_datas['data_sinistro'], sem_data=-1)
sk_data_obito    = atribuir_sk_tempo(df_datas['data_obito'], sem_data=-2)
sk_gravidade     = atribuir_sk_gravidade(df_grav['gravidade_lesao'])

for nome, sk, dim, col in [('data do sinistro', sk_data_sinistro, dim_tempo, 'sk_tempo'),
                           ('data do óbito', sk_data_obito, dim_tempo, 'sk_tempo'),
                           ('gravidade', sk_gravidade, dim_gravidade, 'sk_gravidade')]:
    assert sk.isin(dim[col]).all(), f'{nome}: chave fora da dimensão'
    print(f'{nome:16s} | pessoas com chave: {len(sk)} | -1: {int((sk == -1).sum())} | -2: {int((sk == -2).sum())}')

# o -2 da data do óbito tem que ser exatamente quem não morreu
assert ((sk_data_obito == -2) == ~df_grav['flag_obito'].to_numpy()).all()

data do sinistro | pessoas com chave: 806977 | -1: 0 | -2: 0
data do óbito    | pessoas com chave: 806977 | -1: 0 | -2: 791184
gravidade        | pessoas com chave: 806977 | -1: 70314 | -2: 0


In [69]:
# Diagnóstico DEPOIS: tamanho das dimensões e uso dos membros especiais
depois = pd.DataFrame({
    'linhas': [len(dim_tempo), len(dim_gravidade)],
    'linhas_reais': [int((dim_tempo['sk_tempo'] > 0).sum()), int((dim_gravidade['sk_gravidade'] > 0).sum())],
    'pessoas_-1 (NÃO INFORMADO)': [int((sk_data_sinistro == -1).sum() + (sk_data_obito == -1).sum()),
                                   int((sk_gravidade == -1).sum())],
    'pessoas_-2 (NÃO SE APLICA)': [int((sk_data_obito == -2).sum()), int((sk_gravidade == -2).sum())],
}, index=['dim_tempo', 'dim_gravidade'])
depois

,linhas,linhas_reais,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_tempo,1121,1119,0,791184
dim_gravidade,6,4,70314,0


In [70]:
# gravidade: pessoas por linha da dimensão
pessoas = sk_gravidade.value_counts()
dim_gravidade.assign(pessoas=dim_gravidade['sk_gravidade'].map(pessoas).fillna(0).astype('int64'))

,sk_gravidade,gravidade_lesao,nivel_gravidade,pessoas
0,-1,NÃO INFORMADO,<NA>,70314
1,-2,NÃO SE APLICA,<NA>,0
2,1,ILESO,0,321999
3,2,LEVE,1,357297
4,3,GRAVE,2,41574
5,4,FATAL,3,15793


In [71]:
# só consulta: pessoas envolvidas por tipo de dia (usa a data do sinistro), em 2021–2023
def tipo_de_dia(fim_de_semana: pd.Series, feriado: pd.Series) -> pd.Series:
    """FERIADO vale mais que FIM DE SEMANA; o resto é DIA ÚTIL."""
    tipo = pd.Series('DIA ÚTIL', index=fim_de_semana.index)
    tipo[fim_de_semana.to_numpy(dtype=bool)] = 'FIM DE SEMANA'
    tipo[feriado.to_numpy(dtype=bool)] = 'FERIADO'
    return tipo

ORDEM = ['DIA ÚTIL', 'FIM DE SEMANA', 'FERIADO']
cal = reais[reais['ano'] <= 2023]
dias_cal = tipo_de_dia(cal['fim_de_semana'], cal['feriado']).value_counts().reindex(ORDEM)

do_sinistro = dim_tempo.set_index('sk_tempo').loc[sk_data_sinistro, ['fim_de_semana', 'feriado']].reset_index(drop=True)
pessoas_dia = tipo_de_dia(do_sinistro['fim_de_semana'], do_sinistro['feriado']).value_counts().reindex(ORDEM)

pd.DataFrame({'dias no calendário': dias_cal, 'pessoas': pessoas_dia,
              'pessoas por dia': (pessoas_dia / dias_cal).round(0).astype('int64')})

,dias no calendário,pessoas,pessoas por dia
DIA ÚTIL,760,554750,730
FIM DE SEMANA,308,234889,763
FERIADO,27,17338,642


### 7. Resumo

**O que foi feito:** criação da `dim_tempo` e da `dim_gravidade`, com as colunas do DDL da ETL-18 e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_tempo`: 1.121 linhas, sendo 1.119 dias (01/01/2021 a 24/01/2024, a maior data de óbito) e os 2 membros especiais. A chave é a data no formato `AAAAMMDD`. Tem trimestre, nome do mês, `ano_mes`, dia da semana (1 = segunda ... 7 = domingo), fim de semana e 28 feriados nacionais (9 por ano em 2021–2023 e 1 em janeiro de 2024).
- `dim_gravidade`: 6 linhas, sendo ILESO, LEVE, GRAVE e FATAL (com `nivel_gravidade` de 0 a 3) e os 2 membros especiais.

**Feriados:** só nacionais, como proposto no modelo estrela: os 8 de data fixa mais a Sexta-feira Santa, calculada a partir da Páscoa. Carnaval e Corpus Christi são ponto facultativo e ficam de fora. Os feriados estaduais de SP (como 9 de julho) também ficam de fora; se a equipe quiser, basta acrescentar na lista.

**Membros especiais:** o `NAO DISPONIVEL` de `gravidade_lesao` vai para o -1 (70.314 pessoas, 8,71%). Na data do óbito, o -2 é exatamente quem não morreu (791.184 pessoas). Na `dim_tempo`, os membros especiais têm data e números nulos, e os textos recebem o rótulo, como permite o DDL. O -2 da `dim_gravidade` não é usado; ele existe para seguir a regra de que toda dimensão tem os dois membros.

**O que foi conferido:** o calendário não tem buracos, as chaves são únicas, os textos cabem nos tamanhos do DDL, os dias reais têm todos os campos obrigatórios e as 806.977 pessoas acham chave nas duas datas e na gravidade. A gravidade FATAL coincide com os 15.793 óbitos.

**Para a ETL-23 (tabela fato):** as funções `atribuir_sk_tempo` e `atribuir_sk_gravidade` mostram como ligar cada pessoa às chaves.

---

## Etapa 07b · dim_perfil_vitima e dim_transporte (ETL-20)

- **Etapa:** ETL-20, dim_perfil_vitima e dim_transporte
- **Responsável:** Pedro Wellington
- **Issue:** ETL-20
- **Entrada:** `data/intermediarios/06_limpo.parquet`
- **Saída:** `data/intermediarios/dims/dim_perfil_vitima.parquet` e `dim_transporte.parquet`

Cria as duas dimensões de combinações do modelo estrela, cada uma com chave substituta e os membros especiais -1 e -2.

### 1. Setup

In [72]:
ENTRADA = INTERM / "06_limpo.parquet"              # base limpa unificada

### 2. Carregar a entrada

In [73]:
# colunas de cada dimensão (nomes iguais aos do modelo estrela)
COLUNAS_TRANSPORTE = ['tipo_de_vitima', 'modo_transporte_vitima', 'tipo_veiculo_vitima']
COLUNAS_PERFIL = ['sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal',
                  'nacionalidade', 'grau_de_instrucao']

# lê só as 8 colunas que importam (o parquet tem 34)
df = pd.read_parquet(ENTRADA, columns=COLUNAS_TRANSPORTE + COLUNAS_PERFIL)
print('shape:', df.shape); df.head()

shape: (806977, 8)


,tipo_de_vitima,modo_transporte_vitima,tipo_veiculo_vitima,sexo,faixa_etaria_demografica,faixa_etaria_legal,nacionalidade,grau_de_instrucao
0,PASSAGEIRO,AUTOMOVEL,AUTOMOVEL,FEMININO,30 a 34,30-34,BRASILEIRA,MÉDIO
1,CONDUTOR,AUTOMOVEL,AUTOMOVEL,MASCULINO,30 a 34,30-34,BRASILEIRA,NÃO INFORMADO
2,PEDESTRE,NÃO SE APLICA,NÃO SE APLICA,FEMININO,65 a 69,65-69,BRASILEIRA,SUPERIOR
3,CONDUTOR,AUTOMOVEL,AUTOMOVEL,MASCULINO,25 a 29,25-29,BRASILEIRA,NÃO INFORMADO
4,PASSAGEIRO,NÃO INFORMADO,NÃO INFORMADO,MASCULINO,65 a 69,65-69,BRASILEIRA,SUPERIOR


### 3. Diagnóstico ANTES

In [74]:
antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
tipo_de_vitima,str,0.0,4
modo_transporte_vitima,str,0.0,9
tipo_veiculo_vitima,str,0.0,8
sexo,str,0.0,3
faixa_etaria_demografica,str,0.0,20
faixa_etaria_legal,str,0.0,15
nacionalidade,str,0.0,91
grau_de_instrucao,str,0.0,4


### 4. Criar as dimensões

Cada dimensão guarda uma linha por combinação que existe nos dados, com uma chave `sk_` e os membros -1 e -2.

#### 4.1 Função que monta a dimensão

In [75]:
def criar_dimensao(df: pd.DataFrame, colunas: list, nome_sk: str) -> pd.DataFrame:
    """Uma linha por combinação existente, com chave sk_ e os membros -1 e -2."""
    # 1. combinações distintas das colunas, em ordem
    dim = df[colunas].drop_duplicates().sort_values(colunas).reset_index(drop=True)

    # 2. combinações que são só "NÃO INFORMADO" ou só "NÃO SE APLICA" não viram linha:
    #    elas ficam representadas pelos membros -1 e -2
    so_nao_informado = (dim[colunas] == 'NÃO INFORMADO').all(axis=1)
    so_nao_se_aplica = (dim[colunas] == 'NÃO SE APLICA').all(axis=1)
    dim = dim[~(so_nao_informado | so_nao_se_aplica)].reset_index(drop=True)

    # 3. chave substituta: inteiro de 1 em diante
    dim.insert(0, nome_sk, range(1, len(dim) + 1))

    # 4. membros especiais -1 e -2
    especiais = pd.DataFrame({nome_sk: [-1, -2]})
    for c in colunas:
        especiais[c] = ['NÃO INFORMADO', 'NÃO SE APLICA']

    return pd.concat([especiais, dim], ignore_index=True)

#### 4.2 dim_transporte

In [76]:
dim_transporte = criar_dimensao(df, COLUNAS_TRANSPORTE, 'sk_transporte')
dim_transporte

,sk_transporte,tipo_de_vitima,modo_transporte_vitima,tipo_veiculo_vitima
0,-1,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA
2,1,CONDUTOR,AUTOMOVEL,AUTOMOVEL
3,2,CONDUTOR,BICICLETA,BICICLETA
4,3,CONDUTOR,CAMINHAO,CAMINHAO
5,4,CONDUTOR,MOTOCICLETA,MOTOCICLETA
6,5,CONDUTOR,NÃO INFORMADO,BICICLETA
7,6,CONDUTOR,NÃO INFORMADO,NÃO INFORMADO
8,7,CONDUTOR,ONIBUS,ONIBUS
9,8,CONDUTOR,OUTROS,OUTROS


#### 4.3 dim_perfil_vitima

In [77]:
dim_perfil_vitima = criar_dimensao(df, COLUNAS_PERFIL, 'sk_perfil_vitima')
print(len(dim_perfil_vitima), 'linhas')
dim_perfil_vitima.head(10)

1500 linhas


,sk_perfil_vitima,sexo,faixa_etaria_demografica,faixa_etaria_legal,nacionalidade,grau_de_instrucao
0,-1,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA
2,1,FEMININO,00 a 04,0-17,BOLIVIANA,SUPERIOR
3,2,FEMININO,00 a 04,0-17,BRASILEIRA,BÁSICO
4,3,FEMININO,00 a 04,0-17,BRASILEIRA,MÉDIO
5,4,FEMININO,00 a 04,0-17,BRASILEIRA,NÃO INFORMADO
6,5,FEMININO,00 a 04,0-17,BRASILEIRA,SUPERIOR
7,6,FEMININO,00 a 04,0-17,COLOMBIANA,SUPERIOR
8,7,FEMININO,00 a 04,0-17,HAITIANA,SUPERIOR
9,8,FEMININO,00 a 04,0-17,NÃO INFORMADO,NÃO INFORMADO


### 5. Conferir e salvar

In [78]:
tabelas = [
    ('dim_transporte', dim_transporte, COLUNAS_TRANSPORTE, 'sk_transporte'),
    ('dim_perfil_vitima', dim_perfil_vitima, COLUNAS_PERFIL, 'sk_perfil_vitima'),
]

for nome, dim, colunas, sk in tabelas:
    # a chave não pode se repetir e nenhuma combinação pode aparecer duas vezes
    print(nome, '|', len(dim), 'linhas | sk única:', dim[sk].is_unique,
          '| combinações repetidas:', dim.duplicated(subset=colunas).sum())
    # salva em dims/
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)

dim_transporte | 25 linhas | sk única: True | combinações repetidas: 0
dim_perfil_vitima | 1500 linhas | sk única: True | combinações repetidas: 0


In [79]:
# cada pessoa da base tem que achar uma chave na dimensão
for nome, dim, colunas, sk in tabelas:
    m = df[colunas].merge(dim, on=colunas, how='left')
    print(nome, '| linhas sem chave:', int(m[sk].isna().sum()), '| total:', len(m))

dim_transporte | linhas sem chave: 0 | total: 806977


dim_perfil_vitima | linhas sem chave: 0 | total: 806977


### 6. Resumo

**O que foi feito:** criação das duas dimensões de combinações do modelo estrela, a partir da saída da ETL-16. Cada uma tem uma linha por combinação que existe nos dados, uma chave substituta `sk_` e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_transporte`: 25 linhas (23 combinações de tipo de vítima, modo de transporte e tipo de veículo)
- `dim_perfil_vitima`: 1.500 linhas (1.498 combinações de sexo, faixas etárias, nacionalidade e grau de instrução)

**O que foi conferido:** as chaves são únicas, nenhuma combinação se repete e as 806.977 pessoas encontram uma chave nas duas dimensões.

**Observação:** combinações em que todas as colunas são NÃO INFORMADO não viram linha própria, elas usam o membro -1. A idade exata fica na tabela fato, como no modelo.

**Observação 2:** `nacionalidade` ainda pode ter espaços no fim do texto, que são tratados na ETL-14. Quando essa etapa entrar, o número de combinações pode diminuir um pouco.

---

## Etapa 07c · dim_localidade e dim_via (ETL-21)

- **Etapa:** `07c_dim_localidade_via` (ETL-21)
- **Responsável:** Guilherme Calistrato (`@guicalistrato`)
- **Revisor:** Davi Mello (`@davi01mello`)
- **Issue:** #23 (ETL-21)
- **Entrada:** `data/intermediarios/06_limpo.parquet`
- **Saídas:** `data/intermediarios/dims/dim_localidade.parquet` e `data/intermediarios/dims/dim_via.parquet`

Cria as dimensões `dim_localidade` e `dim_via` do modelo estrela, com chaves substitutas sequenciais `sk_*`, ordenação determinística e os membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`).

### 1. Setup

In [80]:
# Na execução final encadeada (ETL-26), ambas apontam para 06_limpo.parquet
ENTRADA_LOCALIDADE = INTERM / "06_limpo.parquet"
ENTRADA_VIA        = INTERM / "06_limpo.parquet"

COLUNAS_LOCALIDADE = ["cod_ibge", "municipio", "regiao_administrativa"]
COLUNAS_VIA        = ["tipo_via", "local_via"]

### 2. Carregar a entrada

In [81]:
# Lê apenas as colunas necessárias para cada dimensão + id_pessoa para conferir alinhamento
df_loc = pd.read_parquet(ENTRADA_LOCALIDADE, columns=["id_pessoa", *COLUNAS_LOCALIDADE])
df_via = pd.read_parquet(ENTRADA_VIA, columns=["id_pessoa", *COLUNAS_VIA])

# Garante id_pessoa e cod_ibge como inteiros mesmo se a entrada tiver sido salva como texto
df_loc["id_pessoa"] = pd.to_numeric(df_loc["id_pessoa"], errors="raise").astype("int64")
df_loc["cod_ibge"]  = pd.to_numeric(df_loc["cod_ibge"], errors="raise").astype("int64")
df_via["id_pessoa"] = pd.to_numeric(df_via["id_pessoa"], errors="raise").astype("int64")

assert len(df_loc) == len(df_via), "As entradas de localidade e via têm quantidades diferentes de linhas!"
assert df_loc["id_pessoa"].is_unique and df_via["id_pessoa"].is_unique, "id_pessoa duplicado na entrada!"
assert df_loc["id_pessoa"].isin(df_via["id_pessoa"]).all(), "id_pessoa divergente entre as duas entradas!"

print("localidade:", df_loc.shape, "| via:", df_via.shape)
print("mesmas", len(df_loc), "pessoas alinhadas entre as duas entradas")

localidade: (806977, 4) | via: (806977, 3)
mesmas 806977 pessoas alinhadas entre as duas entradas


### 3. Diagnóstico ANTES

In [82]:
antes = pd.concat([
    diagnostico(df_loc[COLUNAS_LOCALIDADE]),
    diagnostico(df_via[COLUNAS_VIA]),
])
antes

,dtype,pct_nulo,n_distintos
cod_ibge,int64,0.0,645
municipio,str,0.0,645
regiao_administrativa,str,0.0,16
tipo_via,str,0.0,3
local_via,str,0.0,3


### 4. `dim_localidade`

Cada linha representa um município paulista identificado por sua chave natural `cod_ibge` (7 dígitos) e sua respectiva `regiao_administrativa`.

**Regras aplicadas:**
- Valida relação 1:1 entre `cod_ibge` e `municipio`, além de 1 única `regiao_administrativa` por município.
- Confere ausência de `D OESTE` / `D ALHO` sem apóstrofo (garantindo a limpeza da ETL-14).
- Ordena por `cod_ibge` crescente antes de numerar `sk_localidade` (`1..645`).
- Inclui os membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`) com `cod_ibge = <NA>` (`Int64`), compatível com `cod_ibge INTEGER NULL UNIQUE` no DDL (`etl_star_schema.sql`).

In [83]:
def criar_dim_localidade(df: pd.DataFrame) -> pd.DataFrame:
    """Cria a dim_localidade ordenada por cod_ibge com membros especiais -1 e -2."""
    assert df[COLUNAS_LOCALIDADE].isna().sum().sum() == 0, "Há nulos nas colunas de localidade!"

    sem_apostrofo = df["municipio"].str.contains(r"\bD (?:OESTE|ALHO)\b", regex=True, na=False)
    assert not sem_apostrofo.any(), "Município sem apóstrofo detectado (verifique saída da ETL-14)!"

    mun_por_ibge = df.groupby("cod_ibge")["municipio"].nunique()
    reg_por_ibge = df.groupby("cod_ibge")["regiao_administrativa"].nunique()
    ibge_por_mun = df.groupby("municipio")["cod_ibge"].nunique()
    assert (mun_por_ibge == 1).all(), "Há cod_ibge associado a mais de um município!"
    assert (reg_por_ibge == 1).all(), "Há cod_ibge associado a mais de uma região administrativa!"
    assert (ibge_por_mun == 1).all(), "Há município associado a mais de um cod_ibge!"

    dim = (
        df[COLUNAS_LOCALIDADE]
        .drop_duplicates()
        .sort_values("cod_ibge", ascending=True)
        .reset_index(drop=True)
    )
    assert dim["cod_ibge"].is_unique and dim["municipio"].is_unique
    assert dim["cod_ibge"].is_monotonic_increasing, "dim_localidade não está ordenada por cod_ibge!"

    dim.insert(0, "sk_localidade", range(1, len(dim) + 1))

    especiais = pd.DataFrame({
        "sk_localidade": [-1, -2],
        "cod_ibge": pd.array([pd.NA, pd.NA], dtype="Int64"),
        "municipio": ["NÃO INFORMADO", "NÃO SE APLICA"],
        "regiao_administrativa": ["NÃO INFORMADO", "NÃO SE APLICA"],
    })
    dim["cod_ibge"] = dim["cod_ibge"].astype("Int64")
    out = pd.concat([especiais, dim], ignore_index=True)
    out["sk_localidade"] = out["sk_localidade"].astype("int64")
    return out

dim_localidade = criar_dim_localidade(df_loc)
print("dim_localidade shape:", dim_localidade.shape)
dim_localidade.head(10)

dim_localidade shape: (647, 4)


,sk_localidade,cod_ibge,municipio,regiao_administrativa
0,-1,<NA>,NÃO INFORMADO,NÃO INFORMADO
1,-2,<NA>,NÃO SE APLICA,NÃO SE APLICA
2,1,3500105,ADAMANTINA,PRESIDENTE PRUDENTE
3,2,3500204,ADOLFO,SÃO JOSÉ DO RIO PRETO
4,3,3500303,AGUAI,CAMPINAS
5,4,3500402,AGUAS DA PRATA,CAMPINAS
6,5,3500501,AGUAS DE LINDOIA,CAMPINAS
7,6,3500550,AGUAS DE SANTA BARBARA,SOROCABA
8,7,3500600,AGUAS DE SAO PEDRO,CAMPINAS
9,8,3500709,AGUDOS,BAURU


### 5. `dim_via`

Dimensão de combinações distintas de `tipo_via` e `local_via`.

**Regras aplicadas:**
- Valida que `tipo_via` e `local_via` já passaram pelo tratamento de nulos e `'NAO DISPONIVEL'` da ETL-16 (zero `NaN` e zero `'NAO DISPONIVEL'`).
- A combinação em que **ambas** as colunas são `'NÃO INFORMADO'` (`('NÃO INFORMADO', 'NÃO INFORMADO')`) não recebe `sk_via > 0`, sendo absorvida pelo membro especial **`-1`** para respeitar `UNIQUE (tipo_via, local_via)` do DDL.
- Combinações parciais (onde apenas uma coluna é `'NÃO INFORMADO'`, como `('VIAS URBANAS', 'NÃO INFORMADO')`) recebem `sk_via > 0` para preservar a informação conhecida.
- Ordena por `['tipo_via', 'local_via']` antes de numerar `sk_via` (`1..8`).

In [84]:
def criar_dim_via(df: pd.DataFrame) -> pd.DataFrame:
    """Cria a dim_via com uma linha por combinação válida (tipo_via, local_via) e membros -1 e -2."""
    assert df[COLUNAS_VIA].isna().sum().sum() == 0, "Há nulos em tipo_via/local_via (verifique ETL-16)!"
    for col in COLUNAS_VIA:
        assert not (df[col] == "NAO DISPONIVEL").any(), f"'{col}' ainda contém 'NAO DISPONIVEL' (verifique ETL-16)!"

    dim = (
        df[COLUNAS_VIA]
        .drop_duplicates()
        .sort_values(COLUNAS_VIA, ascending=True)
        .reset_index(drop=True)
    )

    # Combinações totalmente desconhecidas ou não aplicáveis ficam nos membros -1 e -2
    so_nao_informado = (dim[COLUNAS_VIA] == "NÃO INFORMADO").all(axis=1)
    so_nao_se_aplica = (dim[COLUNAS_VIA] == "NÃO SE APLICA").all(axis=1)
    dim = dim[~(so_nao_informado | so_nao_se_aplica)].reset_index(drop=True)

    assert not dim.duplicated(subset=COLUNAS_VIA).any()
    assert dim[COLUNAS_VIA].apply(tuple, axis=1).is_monotonic_increasing, "dim_via não está ordenada!"
    dim.insert(0, "sk_via", range(1, len(dim) + 1))

    especiais = pd.DataFrame({
        "sk_via": [-1, -2],
        "tipo_via": ["NÃO INFORMADO", "NÃO SE APLICA"],
        "local_via": ["NÃO INFORMADO", "NÃO SE APLICA"],
    })
    out = pd.concat([especiais, dim], ignore_index=True)
    out["sk_via"] = out["sk_via"].astype("int64")
    return out

dim_via = criar_dim_via(df_via)
print("dim_via shape:", dim_via.shape)
dim_via

dim_via shape: (10, 3)


,sk_via,tipo_via,local_via
0,-1,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA
2,1,ESTRADAS E RODOVIAS,NÃO INFORMADO
3,2,ESTRADAS E RODOVIAS,PRIVADO
4,3,ESTRADAS E RODOVIAS,PUBLICO
5,4,NÃO INFORMADO,PRIVADO
6,5,NÃO INFORMADO,PUBLICO
7,6,VIAS URBANAS,NÃO INFORMADO
8,7,VIAS URBANAS,PRIVADO
9,8,VIAS URBANAS,PUBLICO


### 6. Conferir, validar cobertura na base e salvar

Além das validações de unicidade, ordenação e limites de tamanho do DDL (`etl_star_schema.sql`), esta seção define as funções `atribuir_sk_localidade` e `atribuir_sk_via` para reutilização na construção da tabela fato (`ETL-23`).

In [85]:
def atribuir_sk_localidade(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Retorna a sk_localidade de cada linha a partir de cod_ibge (-1 se ausente). Reutilizável na ETL-23."""
    reais = dim[dim["sk_localidade"] > 0].copy()
    mapa = reais.set_index(reais["cod_ibge"].astype("int64"))["sk_localidade"]
    codigos = pd.to_numeric(df["cod_ibge"], errors="coerce")
    return codigos.map(mapa).fillna(-1).astype("int64")

def atribuir_sk_via(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Retorna a sk_via de cada linha a partir do par (tipo_via, local_via) (-1 se ausente). Reutilizável na ETL-23."""
    chaves = df[COLUNAS_VIA].fillna("NÃO INFORMADO").replace("NAO DISPONIVEL", "NÃO INFORMADO")
    m = chaves.merge(dim, on=COLUNAS_VIA, how="left")
    return pd.Series(m["sk_via"].fillna(-1).astype("int64").to_numpy(), index=df.index)

validacoes = [
    ("dim_localidade", dim_localidade, "sk_localidade", ["cod_ibge"], {"municipio": 60, "regiao_administrativa": 40}),
    ("dim_via", dim_via, "sk_via", COLUNAS_VIA, {"tipo_via": 30, "local_via": 20}),
]

for nome, dim, sk, chave_natural, limites in validacoes:
    reais = dim[dim[sk] > 0]
    assert dim[sk].is_unique, f"{nome}: {sk} repetida!"
    assert list(reais[sk]) == list(range(1, len(reais) + 1)), f"{nome}: {sk} não sequencial!"
    assert set(dim.loc[dim[sk] < 0, sk]) == {-1, -2}, f"{nome}: membros -1 e -2 ausentes!"
    assert not reais.duplicated(subset=chave_natural).any(), f"{nome}: chave natural duplicada nos reais!"
    if nome == "dim_via":
        assert not dim.duplicated(subset=chave_natural).any(), "dim_via: combinação duplicada considerando especiais!"
    else:
        assert dim["municipio"].is_unique, "dim_localidade: municipio duplicado!"
        assert dim.loc[dim[sk] < 0, "cod_ibge"].isna().all(), "dim_localidade: especiais devem ter cod_ibge nulo!"
    for col, lim in limites.items():
        max_len = int(dim[col].str.len().max())
        assert max_len <= lim, f"{nome}.{col} excede VARCHAR({lim}): {max_len}"
    dim.to_parquet(DIMS / f"{nome}.parquet", index=False)
    print(f"{nome} | {len(dim)} linhas | {sk} única: True | duplicatas chave natural: 0 | salva em {DIMS / f'{nome}.parquet'}")

dim_localidade | 647 linhas | sk_localidade única: True | duplicatas chave natural: 0 | salva em C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\dims\dim_localidade.parquet
dim_via | 10 linhas | sk_via única: True | duplicatas chave natural: 0 | salva em C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\dims\dim_via.parquet


In [86]:
# Confere que todas as 806.977 pessoas encontram chave válida nas duas dimensões
df_loc["sk_localidade"] = atribuir_sk_localidade(df_loc, dim_localidade)
df_via["sk_via"] = atribuir_sk_via(df_via, dim_via)

cont_loc = df_loc["sk_localidade"].value_counts()
cont_via = df_via["sk_via"].value_counts()

assert cont_loc.sum() == len(df_loc) and (df_loc["sk_localidade"] > 0).all()
assert cont_via.sum() == len(df_via) and df_via["sk_via"].isin(dim_via["sk_via"]).all()

print("pessoas mapeadas | dim_localidade:", int(cont_loc.sum()), "| dim_via:", int(cont_via.sum()))

depois = pd.DataFrame({
    "linhas_dimensao": [len(dim_localidade), len(dim_via)],
    "membros_reais": [int((dim_localidade["sk_localidade"] > 0).sum()), int((dim_via["sk_via"] > 0).sum())],
    "pessoas_-1 (NÃO INFORMADO)": [int(cont_loc.get(-1, 0)), int(cont_via.get(-1, 0))],
    "pessoas_-2 (NÃO SE APLICA)": [int(cont_loc.get(-2, 0)), int(cont_via.get(-2, 0))],
}, index=["dim_localidade", "dim_via"])
depois

pessoas mapeadas | dim_localidade: 806977 | dim_via: 806977


,linhas_dimensao,membros_reais,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_localidade,647,645,0,0
dim_via,10,8,663,0


In [87]:
# Distribuição da dim_via com quantidade de pessoas por combinação
dim_via.assign(pessoas=dim_via["sk_via"].map(cont_via).fillna(0).astype("int64"))

,sk_via,tipo_via,local_via,pessoas
0,-1,NÃO INFORMADO,NÃO INFORMADO,663
1,-2,NÃO SE APLICA,NÃO SE APLICA,0
2,1,ESTRADAS E RODOVIAS,NÃO INFORMADO,19757
3,2,ESTRADAS E RODOVIAS,PRIVADO,150
4,3,ESTRADAS E RODOVIAS,PUBLICO,113686
5,4,NÃO INFORMADO,PRIVADO,7
6,5,NÃO INFORMADO,PUBLICO,303
7,6,VIAS URBANAS,NÃO INFORMADO,107156
8,7,VIAS URBANAS,PRIVADO,4218
9,8,VIAS URBANAS,PUBLICO,561037


In [88]:
# Top 10 municípios com mais registros na dim_localidade
dim_localidade.assign(
    pessoas=dim_localidade["sk_localidade"].map(cont_loc).fillna(0).astype("int64")
).sort_values("pessoas", ascending=False).head(10)

,sk_localidade,cod_ibge,municipio,regiao_administrativa,pessoas
564,563,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,186723
489,488,3543402,RIBEIRAO PRETO,RIBEIRÃO PRETO,21395
110,109,3509502,CAMPINAS,CAMPINAS,20429
214,213,3518800,GUARULHOS,METROPOLITANA DE SÃO PAULO,19452
558,557,3549805,SAO JOSE DO RIO PRETO,SÃO JOSÉ DO RIO PRETO,16521
583,582,3552205,SOROCABA,SOROCABA,14786
546,545,3548708,SAO BERNARDO DO CAMPO,METROPOLITANA DE SÃO PAULO,14494
536,535,3547809,SANTO ANDRE,METROPOLITANA DE SÃO PAULO,11767
295,294,3525904,JUNDIAI,CAMPINAS,11090
559,558,3549904,SAO JOSE DOS CAMPOS,SÃO JOSÉ DOS CAMPOS,10956


### 7. Resumo

**O que foi feito:** criação das dimensões `dim_localidade` e `dim_via` do modelo estrela, com chaves substitutas sequenciais (`sk_*`), ordenação determinística e membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Entradas:** `03_texto.parquet` (`ETL-14`) para `dim_localidade` (municípios com apóstrofos `D'OESTE`/`D'ALHO` restaurados) e `05_nulos_categorias.parquet` (`ETL-16`) para `dim_via` (`NAO DISPONIVEL` e nulos já convertidos para `NÃO INFORMADO`). Na execução final encadeada (`ETL-26`), ambas passam a ler `06_limpo.parquet`.

**Resultados:**
- `dim_localidade`: **647 linhas** (`645` municípios ordenados por `cod_ibge` + membros `-1` e `-2` com `cod_ibge` nulo em `Int64`).
- `dim_via`: **10 linhas** (`8` combinações reais ordenadas por `(tipo_via, local_via)` + membros `-1` e `-2`).

**Decisão sobre `dim_via` (`('NÃO INFORMADO', 'NÃO INFORMADO')`):** nos dados existem 9 combinações de `(tipo_via, local_via)`, mas a combinação dupla `('NÃO INFORMADO', 'NÃO INFORMADO')` (`663` pessoas) é direcionada ao membro `-1` para preservar a restrição `UNIQUE (tipo_via, local_via)` do DDL (`etl_star_schema.sql`). Por isso a dimensão tem `10` linhas (em vez de `11` estimadas inicialmente no modelo lógico).

**O que foi conferido:**
- relação 1:1 entre `cod_ibge` e `municipio`, e 1 única `regiao_administrativa` por município;
- chaves `sk_localidade` e `sk_via` únicas e sequenciais, sem duplicatas nas chaves naturais e dentro dos limites de `VARCHAR` do DDL;
- todas as `806.977` pessoas da base recebem chave válida nas duas dimensões (`0` sem chave).

**Para a ETL-23 (tabela fato):** o notebook disponibiliza as funções `atribuir_sk_localidade` e `atribuir_sk_via` para mapear cada linha da base limpa às chaves das duas dimensões.

---

## Etapa 07d · dim_profissao e dim_local_obito (ETL-22)

- **Etapa:** ETL-22, dim_profissao e dim_local_obito
- **Responsável:** Davi Mello
- **Issue:** ETL-22
- **Entrada:** `data/intermediarios/06_limpo.parquet`
- **Saídas:** `data/intermediarios/dims/dim_profissao.parquet` e `dim_local_obito.parquet`

Cria as duas últimas dimensões descritivas do modelo estrela, cada uma com chave substituta e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA).

### 1. Setup

In [89]:
# Na execução final as duas variáveis apontam para o mesmo arquivo (a saída da etapa anterior)
ENTRADA_PROFISSAO = INTERM / "06_limpo.parquet"
ENTRADA_LOCAL     = INTERM / "06_limpo.parquet"

### 2. Carregar a entrada

In [90]:
# lê só as colunas que importam (os parquets têm 34)
df_prof  = pd.read_parquet(ENTRADA_PROFISSAO, columns=['id_pessoa', 'profissao', 'grupo_profissao'])
df_local = pd.read_parquet(ENTRADA_LOCAL, columns=['id_pessoa', 'local_obito', 'flag_obito'])
print('profissão:', df_prof.shape, '| local do óbito:', df_local.shape)

# os dois arquivos têm que ser da mesma base: mesmas pessoas
assert len(df_prof) == len(df_local) and df_prof['id_pessoa'].is_unique
assert df_prof['id_pessoa'].isin(df_local['id_pessoa']).all()
print('mesmas', len(df_prof), 'pessoas nos dois arquivos')

profissão: (806977, 3) | local do óbito: (806977, 3)
mesmas 806977 pessoas nos dois arquivos


### 3. Diagnóstico ANTES

In [91]:
antes = pd.concat([
    diagnostico(df_prof[['profissao', 'grupo_profissao']]),
    diagnostico(df_local[['local_obito']]),
])
antes

,dtype,pct_nulo,n_distintos
profissao,str,0.0,561
grupo_profissao,str,0.0,13
local_obito,str,0.0,5


### 4. dim_local_obito

O local do óbito vem da ETL-16: quem não morreu está como `NÃO SE APLICA` e o óbito sem local informado como `NÃO INFORMADO`. Sobram só os três locais reais, que viram as linhas da dimensão.

In [92]:
# como o local do óbito chega da ETL-16
print(df_local['local_obito'].value_counts(dropna=False))

# NÃO SE APLICA tem que ser exatamente quem não morreu
nao_se_aplica = df_local['local_obito'] == 'NÃO SE APLICA'
assert (nao_se_aplica == ~df_local['flag_obito']).all()
print('\nóbitos (flag_obito):', int(df_local['flag_obito'].sum()))

local_obito
NÃO SE APLICA               791184
ESTABELECIMENTO DE SAUDE      7940
VIA                           7214
OUTROS                         534
NÃO INFORMADO                  105
Name: count, dtype: int64

óbitos (flag_obito): 15793


In [93]:
LOCAIS = ['ESTABELECIMENTO DE SAUDE', 'OUTROS', 'VIA']

def criar_dim_local_obito(local: pd.Series) -> pd.DataFrame:
    """Uma linha por local de óbito, com chave sk_ e os membros -1 e -2."""
    valores = sorted(local[~local.isin(['NÃO INFORMADO', 'NÃO SE APLICA'])].unique())

    # só aceita os locais conhecidos, para nunca criar uma linha de lixo na dimensão
    assert set(valores) <= set(LOCAIS), f'local de óbito novo: {valores}'

    dim = pd.DataFrame({'sk_local_obito': range(1, len(valores) + 1), 'local_obito': valores})
    especiais = pd.DataFrame({
        'sk_local_obito': [-1, -2],
        'local_obito': ['NÃO INFORMADO', 'NÃO SE APLICA'],
    })
    return pd.concat([especiais, dim], ignore_index=True)

dim_local_obito = criar_dim_local_obito(df_local['local_obito'])
dim_local_obito

,sk_local_obito,local_obito
0,-1,NÃO INFORMADO
1,-2,NÃO SE APLICA
2,1,ESTABELECIMENTO DE SAUDE
3,2,OUTROS
4,3,VIA


### 5. dim_profissao

A dimensão tem uma linha por profissão já padronizada na ETL-17, com o `grupo_profissao`. Antes de criar, falta um cuidado com os nulos.

#### 5.1 Nulos escondidos na profissão

A ETL-17 padronizou a escrita, mas duas grafias de "não informado" da própria fonte continuam como se fossem profissões e caíram no grupo `OUTROS`. Aqui só consulta: nada é alterado.

In [94]:
# grafias que dizem "não informado" ou "sem profissão" (só consulta)
contagem = df_prof['profissao'].value_counts()
suspeitas = contagem[contagem.index.str.contains('INFORMAD|DEFINID', na=False)]
resumo = suspeitas.to_frame('pessoas')
resumo['grupo_profissao'] = [df_prof.loc[df_prof['profissao'] == p, 'grupo_profissao'].iloc[0] for p in resumo.index]
resumo

,pessoas,grupo_profissao
profissao,,
NAO INFORMADO,86198,NÃO INFORMADO
NAO INFORMADA,636,OUTROS
SEM PROFISSAO DEFINIDA,9,OUTROS


#### 5.2 Função que leva os nulos ao membro -1

`NAO INFORMADA` e `NAO INFORMADO` significam o mesmo que o campo vazio. Eles, o vazio e o `NÃO INFORMADO` que a ETL-16 grava passam a usar o membro -1. `SEM PROFISSAO DEFINIDA` (9 pessoas) é uma resposta e continua como profissão.

In [95]:
# grafias de "não informado" que a ETL-17 ainda deixa como profissão
NAO_INFORMADO = ['NAO INFORMADA', 'NAO INFORMADO', 'NÃO INFORMADO']

def profissao_para_dimensao(profissao: pd.Series) -> pd.Series:
    """Nulo e as grafias de "não informado" viram nulo: todos usam o membro -1 da dimensão."""
    return profissao.mask(profissao.isin(NAO_INFORMADO))

# a função não muda nada em quem já tem profissão
chave = profissao_para_dimensao(df_prof['profissao'])
print('nulos antes:', int(df_prof['profissao'].isna().sum()), '| depois:', int(chave.isna().sum()))

nulos antes: 0 | depois: 86834


#### 5.3 Criar a dimensão

In [96]:
def criar_dim_profissao(df: pd.DataFrame) -> pd.DataFrame:
    """Uma linha por profissão padronizada, com seu grupo, chave sk_ e os membros -1 e -2."""
    base = df[['profissao', 'grupo_profissao']].copy()
    base['profissao'] = profissao_para_dimensao(base['profissao'])

    # quem está sem profissão não vira linha: usa o membro -1
    base = base.dropna(subset=['profissao'])
    dim = base.drop_duplicates().sort_values('profissao').reset_index(drop=True)

    # cada profissão tem que ter um grupo só (senão a mesma chave teria dois grupos)
    assert dim['profissao'].is_unique, 'profissão com mais de um grupo'

    dim.insert(0, 'sk_profissao', range(1, len(dim) + 1))
    especiais = pd.DataFrame({
        'sk_profissao': [-1, -2],
        'profissao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
        'grupo_profissao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
    })
    return pd.concat([especiais, dim], ignore_index=True)

dim_profissao = criar_dim_profissao(df_prof)
print(len(dim_profissao), 'linhas')
dim_profissao.head(8)

561 linhas


,sk_profissao,profissao,grupo_profissao
0,-1,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA
2,1,ACOUGUEIRO,SERVICOS PESSOAIS E DOMESTICOS
3,2,ADESTRADOR DE ANIMAIS,OUTROS
4,3,ADMINISTRADOR,ADMINISTRATIVO
5,4,ADMINISTRADOR DE EMPRESAS,ADMINISTRATIVO
6,5,ADMINISTRADOR DE FAZENDA,ADMINISTRATIVO
7,6,ADMINISTRADOR RURAL,ADMINISTRATIVO


In [97]:
# quantas profissões tem cada grupo (12 grupos)
dim_profissao[dim_profissao['sk_profissao'] > 0]['grupo_profissao'].value_counts()

grupo_profissao
OUTROS                            305
INDUSTRIA E CONSTRUCAO             51
ADMINISTRATIVO                     50
TECNICOS E NIVEL SUPERIOR          40
SEGURANCA E SERVICO PUBLICO        23
SAUDE E EDUCACAO                   22
SERVICOS PESSOAIS E DOMESTICOS     18
TRANSPORTE E ENTREGA               17
COMERCIO E VENDAS                  17
AGRICULTURA E CAMPO                 6
SEM OCUPACAO                        6
AUTONOMO E EMPRESARIO               4
Name: count, dtype: int64

### 6. Conferir e salvar

Quatro conferências: a chave não se repete, nenhum nome passa do tamanho previsto no modelo, toda pessoa acha uma chave e os membros especiais têm o tamanho esperado.

In [98]:
def atribuir_sk_profissao(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Chave de cada pessoa na dim_profissao (-1 quando não há profissão). A ETL-23 pode reutilizar."""
    mapa = dim[dim['sk_profissao'] > 0].set_index('profissao')['sk_profissao']
    return profissao_para_dimensao(df['profissao']).map(mapa).fillna(-1).astype('int64')

def atribuir_sk_local_obito(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Chave de cada pessoa na dim_local_obito. A ETL-23 pode reutilizar."""
    mapa = dim.set_index('local_obito')['sk_local_obito']
    return df['local_obito'].map(mapa).astype('int64')

tabelas = [
    ('dim_profissao', dim_profissao, 'sk_profissao', ['profissao'], {'profissao': 80, 'grupo_profissao': 40}),
    ('dim_local_obito', dim_local_obito, 'sk_local_obito', ['local_obito'], {'local_obito': 30}),
]
for nome, dim, sk, natural, limites in tabelas:
    print(nome, '|', len(dim), 'linhas | sk única:', dim[sk].is_unique,
          '| nomes repetidos:', int(dim.duplicated(subset=natural).sum()))
    for col, limite in limites.items():
        assert dim[col].str.len().max() <= limite, f'{col} maior que {limite}'
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)

dim_profissao | 561 linhas | sk única: True | nomes repetidos: 0
dim_local_obito | 5 linhas | sk única: True | nomes repetidos: 0


In [99]:
# cada pessoa tem que achar uma chave nas duas dimensões
df_prof['sk_profissao'] = atribuir_sk_profissao(df_prof, dim_profissao)
df_local['sk_local_obito'] = atribuir_sk_local_obito(df_local, dim_local_obito)

pessoas_prof = df_prof['sk_profissao'].value_counts()
pessoas_local = df_local['sk_local_obito'].value_counts()
assert pessoas_prof.sum() == len(df_prof) and pessoas_local.sum() == len(df_local)
print('pessoas com chave | profissão:', int(pessoas_prof.sum()), '| local do óbito:', int(pessoas_local.sum()))

# o grupo da pessoa (ETL-17) é o mesmo da dimensão, menos nas que foram para -1
com_chave = df_prof[df_prof['sk_profissao'] > 0].merge(dim_profissao, on='sk_profissao', suffixes=('', '_dim'))
print('grupos diferentes da dimensão:', int((com_chave['grupo_profissao'] != com_chave['grupo_profissao_dim']).sum()))

pessoas com chave | profissão: 806977 | local do óbito: 806977
grupos diferentes da dimensão: 0


In [100]:
# membros especiais: quantas pessoas usam cada um
depois = pd.DataFrame({
    'linhas': [len(dim_profissao), len(dim_local_obito)],
    'pessoas_-1 (NÃO INFORMADO)': [int(pessoas_prof.get(-1, 0)), int(pessoas_local.get(-1, 0))],
    'pessoas_-2 (NÃO SE APLICA)': [int(pessoas_prof.get(-2, 0)), int(pessoas_local.get(-2, 0))],
}, index=['dim_profissao', 'dim_local_obito'])
depois

,linhas,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_profissao,561,86834,0
dim_local_obito,5,105,791184


In [101]:
# local do óbito: pessoas por linha da dimensão (-2 = quem não morreu)
dim_local_obito.assign(pessoas=dim_local_obito['sk_local_obito'].map(pessoas_local))

,sk_local_obito,local_obito,pessoas
0,-1,NÃO INFORMADO,105
1,-2,NÃO SE APLICA,791184
2,1,ESTABELECIMENTO DE SAUDE,7940
3,2,OUTROS,534
4,3,VIA,7214


In [102]:
# profissão: de onde vêm as pessoas do membro -1 (grupo que a ETL-17 tinha dado a elas)
df_prof[df_prof['sk_profissao'] == -1]['grupo_profissao'].value_counts()

grupo_profissao
NÃO INFORMADO    86198
OUTROS             636
Name: count, dtype: int64

### 7. Resumo

**O que foi feito:** criação das duas últimas dimensões descritivas do modelo estrela, cada uma com chave substituta `sk_` e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_profissao`: 561 linhas (559 profissões padronizadas, cada uma com seu `grupo_profissao`, mais os 2 membros especiais).
- `dim_local_obito`: 5 linhas (ESTABELECIMENTO DE SAUDE, OUTROS e VIA, mais os 2 membros especiais).

**Nulos escondidos:** `NAO INFORMADA` (636) e `NAO INFORMADO` (298) são "não informado" escrito como profissão. A ETL-17 os deixou no grupo `OUTROS`. Aqui eles usam o membro -1, que passou a ter 86.834 pessoas (85.900 campos vazios e 934 dessas grafias). `SEM PROFISSAO DEFINIDA` (9 pessoas) continua como profissão.

**Local do óbito:** 791.184 pessoas usam o -2 (não morreram) e 105 usam o -1 (morreram, sem local). Os 15.793 óbitos se dividem em 7.940 em estabelecimento de saúde, 7.214 na via e 534 em outros locais.

**O que foi conferido:** as chaves são únicas, nenhum nome se repete, cada profissão tem um grupo só e as 806.977 pessoas encontram uma chave nas duas dimensões. O membro -2 da `dim_profissao` não é usado (todos têm profissão ou não informada); ele existe só para seguir a regra de que toda dimensão tem os dois membros.

**Para a ETL-23 (tabela fato):** as funções `atribuir_sk_profissao` e `atribuir_sk_local_obito` mostram como ligar cada pessoa à chave. A profissão precisa passar por `profissao_para_dimensao` antes do join, porque a grafia `NAO INFORMADO` não está na dimensão.

**Observação:** o ideal é que a ETL-17 trate `NAO INFORMADA` e `NAO INFORMADO` como nulo. Se isso for feito lá, esta etapa continua valendo sem mudar, porque a função aceita as duas formas.

---

## Etapa 08 · Tabela fato (ETL-23)

- **Etapa:** ETL-23, tabela fato (`fato_pessoa_sinistro`)
- **Responsável:** Davi Mello
- **Issue:** ETL-23
- **Entradas:** `data/intermediarios/06_limpo.parquet` e as 8 dimensões em `data/intermediarios/dims/` (ETL-19 a ETL-22)
- **Saída:** `data/intermediarios/fato.parquet`

Monta a fato no grão de **uma pessoa em um sinistro**: cada grupo de colunas descritivas vira a chave `sk_` da sua dimensão, com duas chaves para a `dim_tempo` (data do sinistro e data do óbito). As colunas seguem o DDL da ETL-18 (`sql/ddl/etl_star_schema.sql`).

### 1. Setup

In [103]:
SAIDA  = INTERM / "fato.parquet"               # resultado desta etapa

COLUNAS_TRANSPORTE = ['tipo_de_vitima', 'modo_transporte_vitima', 'tipo_veiculo_vitima']
COLUNAS_PERFIL = ['sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'nacionalidade', 'grau_de_instrucao']
COLUNAS_VIA = ['tipo_via', 'local_via']

# Cada grupo de colunas vem do mesmo arquivo que gerou a sua dimensão, para os valores baterem.
# Na execução final (ETL-26) todos apontam para o mesmo arquivo: a saída da última etapa de limpeza.
FONTES = {
    'ids (ETL-13)':        (INTERM / '06_limpo.parquet', ['id_sinistro', 'id_veiculo']),
    'datas (ETL-15)':      (INTERM / '06_limpo.parquet', ['data_sinistro', 'data_obito', 'dias_ate_obito']),
    'localidade (ETL-14)': (INTERM / '06_limpo.parquet', ['cod_ibge']),
    'categorias (ETL-16)': (INTERM / '06_limpo.parquet',
                            ['gravidade_lesao', 'local_obito', 'idade', 'flag_obito',
                             *COLUNAS_TRANSPORTE, *COLUNAS_PERFIL, *COLUNAS_VIA]),
    'profissão (ETL-17)':  (INTERM / '06_limpo.parquet', ['profissao']),
}

### 2. Carregar a entrada

#### 2.1 Base limpa (juntando os grupos pelo `id_pessoa`)

In [104]:
def carregar_base(fontes: dict) -> pd.DataFrame:
    """Lê cada grupo de colunas do seu arquivo e junta tudo pelo id_pessoa, sem perder nem duplicar linhas."""
    base = None
    for nome, (arquivo, colunas) in fontes.items():
        parte = pd.read_parquet(arquivo, columns=['id_pessoa', *colunas])
        parte['id_pessoa'] = pd.to_numeric(parte['id_pessoa'], errors='raise').astype('int64')
        assert parte['id_pessoa'].is_unique, f'{nome}: id_pessoa repetido'
        if base is None:
            base = parte
        else:
            n = len(base)
            base = base.merge(parte, on='id_pessoa', how='inner', validate='one_to_one')
            assert len(base) == n == len(parte), f'{nome}: pessoas diferentes das outras fontes'
        print(f'{nome:20s} | {arquivo.name:30s} | {len(parte)} linhas')
    return base

base = carregar_base(FONTES)
print('\nbase:', base.shape)

ids (ETL-13)         | 06_limpo.parquet               | 806977 linhas


datas (ETL-15)       | 06_limpo.parquet               | 806977 linhas


localidade (ETL-14)  | 06_limpo.parquet               | 806977 linhas


categorias (ETL-16)  | 06_limpo.parquet               | 806977 linhas


profissão (ETL-17)   | 06_limpo.parquet               | 806977 linhas

base: (806977, 22)


#### 2.2 As 8 dimensões

In [105]:
NOMES_DIMS = ['dim_tempo', 'dim_gravidade', 'dim_perfil_vitima', 'dim_transporte',
              'dim_localidade', 'dim_via', 'dim_profissao', 'dim_local_obito']
dims = {nome: pd.read_parquet(DIMS / f'{nome}.parquet') for nome in NOMES_DIMS}

pd.DataFrame({'linhas': {n: len(d) for n, d in dims.items()},
              'membros -1 e -2': {n: int(d.iloc[:, 0].isin([-1, -2]).sum()) for n, d in dims.items()}})

,linhas,membros -1 e -2
dim_tempo,1121,2
dim_gravidade,6,2
dim_perfil_vitima,1500,2
dim_transporte,25,2
dim_localidade,647,2
dim_via,10,2
dim_profissao,561,2
dim_local_obito,5,2


### 3. Diagnóstico ANTES

In [106]:
antes = diagnostico(base)
antes

,dtype,pct_nulo,n_distintos
id_pessoa,int64,0.00,806977
id_sinistro,int64,0.00,402695
id_veiculo,Int64,21.04,554294
data_sinistro,datetime64[us],0.00,1095
data_obito,datetime64[us],98.04,1109
dias_ate_obito,Int64,98.04,31
cod_ibge,int64,0.00,645
gravidade_lesao,str,0.00,5
local_obito,str,0.00,5
idade,Int64,1.04,103


In [107]:
# números da base limpa que a fato tem que repetir
BASE_LINHAS    = len(base)
BASE_SINISTROS = base['id_sinistro'].nunique()
BASE_OBITOS    = int(base['flag_obito'].sum())
print('pessoas:', BASE_LINHAS, '| sinistros:', BASE_SINISTROS, '| óbitos:', BASE_OBITOS)

pessoas: 806977 | sinistros: 402695 | óbitos: 15793


### 4. Trocar as colunas pelas chaves

Cada função devolve a chave de cada pessoa na dimensão e **para com erro** se alguma pessoa não achar chave. Assim nenhum valor inesperado vira -1 em silêncio. O valor ausente só vai para -1 (NÃO INFORMADO) ou -2 (NÃO SE APLICA) pelas regras de cada dimensão.

#### 4.1 Funções de busca

In [108]:
def conferir_chave(sk: pd.Series, nome: str) -> pd.Series:
    """Para com erro se alguma pessoa ficou sem chave; senão devolve a chave como inteiro."""
    sem_chave = sk.isna()
    assert not sem_chave.any(), f'{nome}: {int(sem_chave.sum())} pessoas sem chave'
    return sk.astype('int64')

def sk_por_combinacao(df: pd.DataFrame, dim: pd.DataFrame, colunas: list, sk: str) -> pd.Series:
    """Dimensões de combinações (perfil, transporte, via): busca pela combinação das colunas.
    As combinações só de NÃO INFORMADO ou só de NÃO SE APLICA acham os membros -1 e -2 pelo rótulo."""
    chaves = df[colunas].fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    m = chaves.merge(dim[[*colunas, sk]], on=colunas, how='left', validate='many_to_one')
    return conferir_chave(pd.Series(m[sk].to_numpy(), index=df.index), sk)

def sk_tempo(datas: pd.Series, sem_data: int) -> pd.Series:
    """dim_tempo: a chave é a data no formato AAAAMMDD; quem não tem data recebe sem_data."""
    if not pd.api.types.is_datetime64_any_dtype(datas):
        datas = pd.to_datetime(datas, format='%d/%m/%Y', errors='raise')
    sk = datas.dt.year * 10000 + datas.dt.month * 100 + datas.dt.day
    return sk.fillna(sem_data).astype('int64')

NIVEIS = {'ILESO': 0, 'LEVE': 1, 'GRAVE': 2, 'FATAL': 3}
def sk_gravidade(gravidade: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_gravidade: busca pelo nome; NAO DISPONIVEL e vazio vão para -1."""
    mapa = dim.set_index('gravidade_lesao')['sk_gravidade']
    nome = gravidade.fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    return conferir_chave(nome.map(mapa), 'sk_gravidade')

def sk_localidade(cod_ibge: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_localidade: busca pelo código IBGE."""
    reais = dim[dim['sk_localidade'] > 0]
    mapa = reais.set_index(reais['cod_ibge'].astype('int64'))['sk_localidade']
    return conferir_chave(pd.to_numeric(cod_ibge, errors='raise').map(mapa), 'sk_localidade')

NAO_INFORMADO_PROFISSAO = ['NAO INFORMADA', 'NAO INFORMADO', 'NÃO INFORMADO']
def sk_profissao(profissao: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_profissao: vazio e as grafias de "não informado" vão para -1 (regra da ETL-22)."""
    reais = dim[dim['sk_profissao'] > 0]
    mapa = reais.set_index('profissao')['sk_profissao']
    chave = profissao.mask(profissao.isin(NAO_INFORMADO_PROFISSAO))
    sk = chave.map(mapa)
    sk[chave.isna()] = -1
    return conferir_chave(sk, 'sk_profissao')

def sk_local_obito(local: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_local_obito: busca pelo nome (NÃO INFORMADO e NÃO SE APLICA já vêm da ETL-16)."""
    mapa = dim.set_index('local_obito')['sk_local_obito']
    return conferir_chave(local.map(mapa), 'sk_local_obito')

#### 4.2 Montar a fato

In [109]:
def montar_fato(base: pd.DataFrame, dims: dict) -> pd.DataFrame:
    """Uma linha por pessoa: 3 identificadores da fonte, 9 chaves e 4 medidas, na ordem do DDL."""
    fato = pd.DataFrame(index=base.index)

    # dimensões degeneradas (identificadores da fonte)
    fato['id_pessoa'] = base['id_pessoa'].astype('int64')
    fato['id_sinistro'] = base['id_sinistro'].astype('int64')
    fato['id_veiculo'] = base['id_veiculo'].astype('Int64')      # nulo permitido

    # chaves das dimensões
    fato['sk_data_sinistro'] = sk_tempo(base['data_sinistro'], sem_data=-1)
    fato['sk_data_obito'] = sk_tempo(base['data_obito'], sem_data=-2)   # -2: quem não morreu
    fato['sk_gravidade'] = sk_gravidade(base['gravidade_lesao'], dims['dim_gravidade'])
    fato['sk_perfil_vitima'] = sk_por_combinacao(base, dims['dim_perfil_vitima'], COLUNAS_PERFIL, 'sk_perfil_vitima')
    fato['sk_transporte'] = sk_por_combinacao(base, dims['dim_transporte'], COLUNAS_TRANSPORTE, 'sk_transporte')
    fato['sk_localidade'] = sk_localidade(base['cod_ibge'], dims['dim_localidade'])
    fato['sk_via'] = sk_por_combinacao(base, dims['dim_via'], COLUNAS_VIA, 'sk_via')
    fato['sk_profissao'] = sk_profissao(base['profissao'], dims['dim_profissao'])
    fato['sk_local_obito'] = sk_local_obito(base['local_obito'], dims['dim_local_obito'])

    # medidas
    fato['idade'] = base['idade'].astype('Int16')
    fato['dias_ate_obito'] = base['dias_ate_obito'].astype('Int16')
    fato['flag_obito'] = base['flag_obito'].astype('int16')
    fato['qtd_pessoas'] = pd.Series(1, index=base.index, dtype='int16')
    return fato.reset_index(drop=True)

fato = montar_fato(base, dims)
print(fato.shape)
fato.head()

(806977, 16)


,id_pessoa,id_sinistro,id_veiculo,sk_data_sinistro,sk_data_obito,sk_gravidade,sk_perfil_vitima,sk_transporte,sk_localidade,sk_via,sk_profissao,sk_local_obito,idade,dias_ate_obito,flag_obito,qtd_pessoas
0,1979797,2240338,2874666,20210101,-2,1,184,14,585,8,228,-2,31,<NA>,0,1
1,1979798,2240338,2874666,20210101,-2,1,839,1,585,8,385,-2,34,<NA>,0,1
2,1284984,1447800,<NA>,20210101,-2,1,444,22,563,8,210,-2,65,<NA>,0,1
3,563279,1783410,2648403,20210101,-2,3,760,1,563,8,80,-2,26,<NA>,0,1
4,687322,1798856,<NA>,20210101,-2,2,1311,19,532,8,52,-2,66,<NA>,0,1


### 5. Conferências

Seis conferências: os totais batem com a base limpa, toda chave existe na sua dimensão, as regras (`CHECK`) do DDL valem, as medidas batem com as datas, a fato devolve os valores originais quando ligada de volta às dimensões e o total por ano bate com a fonte.

In [110]:
# 1. totais: linhas, pessoas, sinistros e óbitos iguais aos da base limpa
totais = pd.DataFrame({
    'base limpa': [BASE_LINHAS, BASE_LINHAS, BASE_SINISTROS, BASE_OBITOS],
    'fato': [len(fato), fato['id_pessoa'].nunique(), fato['id_sinistro'].nunique(), int(fato['flag_obito'].sum())],
}, index=['linhas', 'pessoas (id_pessoa)', 'sinistros (id_sinistro)', 'óbitos (flag_obito)'])
totais['confere'] = totais['base limpa'] == totais['fato']
assert totais['confere'].all()
totais

,base limpa,fato,confere
linhas,806977,806977,True
pessoas (id_pessoa),806977,806977,True
sinistros (id_sinistro),402695,402695,True
óbitos (flag_obito),15793,15793,True


In [111]:
# 2. integridade referencial: toda chave da fato existe na sua dimensão
FKS = {
    'sk_data_sinistro': ('dim_tempo', 'sk_tempo'),
    'sk_data_obito': ('dim_tempo', 'sk_tempo'),
    'sk_gravidade': ('dim_gravidade', 'sk_gravidade'),
    'sk_perfil_vitima': ('dim_perfil_vitima', 'sk_perfil_vitima'),
    'sk_transporte': ('dim_transporte', 'sk_transporte'),
    'sk_localidade': ('dim_localidade', 'sk_localidade'),
    'sk_via': ('dim_via', 'sk_via'),
    'sk_profissao': ('dim_profissao', 'sk_profissao'),
    'sk_local_obito': ('dim_local_obito', 'sk_local_obito'),
}
for fk, (dim, pk) in FKS.items():
    assert fato[fk].notna().all() and fato[fk].isin(dims[dim][pk]).all(), f'{fk}: chave fora da {dim}'
print('as 9 chaves estrangeiras existem nas dimensões e nenhuma é nula')

as 9 chaves estrangeiras existem nas dimensões e nenhuma é nula


In [112]:
# 3. regras do DDL (as mesmas CHECK que o PostgreSQL vai cobrar na carga)
sem_obito = fato['flag_obito'] == 0
regras = {
    'id_pessoa único (PK)': fato['id_pessoa'].is_unique,
    'ids cabem em INTEGER': bool(fato[['id_pessoa', 'id_sinistro']].max().max() < 2**31
                                 and fato['id_veiculo'].max() < 2**31),
    'idade entre 0 e 120 ou nula': bool(fato['idade'].dropna().between(0, 120).all()),
    'dias_ate_obito entre 0 e 30 ou nulo': bool(fato['dias_ate_obito'].dropna().between(0, 30).all()),
    'flag_obito é 0 ou 1': bool(fato['flag_obito'].isin([0, 1]).all()),
    'qtd_pessoas sempre 1': bool((fato['qtd_pessoas'] == 1).all()),
    'sem óbito: data -2, local -2 e dias nulo': bool((
        (fato.loc[sem_obito, 'sk_data_obito'] == -2) & (fato.loc[sem_obito, 'sk_local_obito'] == -2)
        & fato.loc[sem_obito, 'dias_ate_obito'].isna()).all()),
    'com óbito: data diferente de -2': bool((fato.loc[~sem_obito, 'sk_data_obito'] != -2).all()),
}
regras = pd.Series(regras, name='ok')
assert regras.all()
regras.to_frame()

,ok
id_pessoa único (PK),True
ids cabem em INTEGER,True
idade entre 0 e 120 ou nula,True
dias_ate_obito entre 0 e 30 ou nulo,True
flag_obito é 0 ou 1,True
qtd_pessoas sempre 1,True
"sem óbito: data -2, local -2 e dias nulo",True
com óbito: data diferente de -2,True


In [113]:
# 4. as medidas batem com as datas: dias_ate_obito = data do óbito - data do sinistro
datas_sin = pd.to_datetime(fato['sk_data_sinistro'].astype(str), format='%Y%m%d')
obitos = fato['sk_data_obito'] > 0
datas_obi = pd.to_datetime(fato.loc[obitos, 'sk_data_obito'].astype(str), format='%Y%m%d')
diferenca = (datas_obi - datas_sin[obitos]).dt.days
assert (diferenca == fato.loc[obitos, 'dias_ate_obito'].astype('int64')).all()
print('dias_ate_obito confere com as duas datas em', int(obitos.sum()), 'óbitos')

dias_ate_obito confere com as duas datas em 15793 óbitos


In [114]:
# 5. ida e volta: ligando a fato às dimensões, cada pessoa recupera os valores da base limpa
def normalizado(serie: pd.Series) -> pd.Series:
    """Mesma regra de ausente das dimensões: vazio e NAO DISPONIVEL viram NÃO INFORMADO."""
    return serie.fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')

COMPARAR = [   # (chave na fato, dimensão, colunas da dimensão = colunas da base)
    ('sk_gravidade', 'dim_gravidade', ['gravidade_lesao']),
    ('sk_perfil_vitima', 'dim_perfil_vitima', COLUNAS_PERFIL),
    ('sk_transporte', 'dim_transporte', COLUNAS_TRANSPORTE),
    ('sk_via', 'dim_via', COLUNAS_VIA),
    ('sk_local_obito', 'dim_local_obito', ['local_obito']),
]
divergencias = {}
for fk, nome, colunas in COMPARAR:
    pk = FKS[fk][1]
    volta = fato[[fk]].merge(dims[nome], left_on=fk, right_on=pk, how='left')
    for c in colunas:
        divergencias[c] = int((volta[c].to_numpy() != normalizado(base[c]).to_numpy()).sum())

# localidade (pelo código IBGE) e profissão (com a regra de "não informado" da ETL-22)
volta = fato[['sk_localidade']].merge(dims['dim_localidade'], on='sk_localidade', how='left')
divergencias['cod_ibge'] = int((volta['cod_ibge'].astype('int64').to_numpy() != base['cod_ibge'].to_numpy()).sum())
volta = fato[['sk_profissao']].merge(dims['dim_profissao'], on='sk_profissao', how='left')
prof = base['profissao'].mask(base['profissao'].isin(NAO_INFORMADO_PROFISSAO)).fillna('NÃO INFORMADO')
divergencias['profissao'] = int((volta['profissao'].to_numpy() != prof.to_numpy()).sum())

# datas
divergencias['data_sinistro'] = int((datas_sin.to_numpy() != base['data_sinistro'].to_numpy()).sum())
divergencias['data_obito'] = int((datas_obi.to_numpy() != base.loc[obitos, 'data_obito'].to_numpy()).sum())

divergencias = pd.Series(divergencias, name='pessoas com valor diferente')
assert (divergencias == 0).all()
divergencias.to_frame()

,pessoas com valor diferente
gravidade_lesao,0
sexo,0
faixa_etaria_demografica,0
faixa_etaria_legal,0
nacionalidade,0
grau_de_instrucao,0
tipo_de_vitima,0
modo_transporte_vitima,0
tipo_veiculo_vitima,0
tipo_via,0


In [115]:
# 6. pessoas e óbitos por ano do sinistro, pela dim_tempo (fonte: 260.886 / 266.128 / 279.963)
por_ano = (fato.merge(dims['dim_tempo'][['sk_tempo', 'ano']], left_on='sk_data_sinistro', right_on='sk_tempo')
               .groupby('ano')[['qtd_pessoas', 'flag_obito']].sum()
               .rename(columns={'qtd_pessoas': 'pessoas', 'flag_obito': 'óbitos'}))
assert por_ano['pessoas'].tolist() == [260886, 266128, 279963]
por_ano

,pessoas,óbitos
ano,,
2021,260886,4858
2022,266128,5445
2023,279963,5490


### 6. Diagnóstico DEPOIS

In [116]:
depois = diagnostico(fato)
depois

,dtype,pct_nulo,n_distintos
id_pessoa,int64,0.00,806977
id_sinistro,int64,0.00,402695
id_veiculo,Int64,21.04,554294
sk_data_sinistro,int64,0.00,1095
sk_data_obito,int64,0.00,1110
sk_gravidade,int64,0.00,5
sk_perfil_vitima,int64,0.00,1499
sk_transporte,int64,0.00,24
sk_localidade,int64,0.00,645
sk_via,int64,0.00,9


In [117]:
# uso dos membros especiais em cada chave (pessoas)
pd.DataFrame({
    '-1 (NÃO INFORMADO)': {fk: int((fato[fk] == -1).sum()) for fk in FKS},
    '-2 (NÃO SE APLICA)': {fk: int((fato[fk] == -2).sum()) for fk in FKS},
})

,-1 (NÃO INFORMADO),-2 (NÃO SE APLICA)
sk_data_sinistro,0,0
sk_data_obito,0,791184
sk_gravidade,70314,0
sk_perfil_vitima,3395,0
sk_transporte,98947,0
sk_localidade,0,0
sk_via,663,0
sk_profissao,86834,0
sk_local_obito,105,791184


### 7. Salvar a saída

In [118]:
fato.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA.relative_to(RAIZ), '| shape:', fato.shape)

salvo em: data\intermediarios\fato.parquet | shape: (806977, 16)


### 8. Consulta de teste no esquema estrela (só consulta)

Uma pergunta respondida só com a fato e as dimensões, como será feito no PostgreSQL: letalidade por gravidade registrada e por tipo de dia do sinistro.

In [119]:
tempo = dims['dim_tempo'].set_index('sk_tempo')
dia = tempo.loc[fato['sk_data_sinistro'], ['fim_de_semana', 'feriado']].reset_index(drop=True)
tipo_dia = pd.Series('DIA ÚTIL', index=fato.index)
tipo_dia[dia['fim_de_semana'].to_numpy(dtype=bool)] = 'FIM DE SEMANA'
tipo_dia[dia['feriado'].to_numpy(dtype=bool)] = 'FERIADO'

consulta = (fato.assign(tipo_dia=tipo_dia)
                .groupby('tipo_dia')[['qtd_pessoas', 'flag_obito']].sum()
                .rename(columns={'qtd_pessoas': 'pessoas', 'flag_obito': 'óbitos'}))
consulta['letalidade (%)'] = (consulta['óbitos'] / consulta['pessoas'] * 100).round(2)
consulta.reindex(['DIA ÚTIL', 'FIM DE SEMANA', 'FERIADO'])

,pessoas,óbitos,letalidade (%)
tipo_dia,,,
DIA ÚTIL,554750,9243,1.67
FIM DE SEMANA,234889,6039,2.57
FERIADO,17338,511,2.95


### 9. Resumo

**O que foi feito:** montagem da `fato_pessoa_sinistro`, com uma linha por pessoa envolvida em um sinistro. As 23 colunas aproveitadas da fonte viraram 3 identificadores (`id_pessoa`, `id_sinistro` e `id_veiculo`), 9 chaves para as 8 dimensões (duas para a `dim_tempo`: data do sinistro e data do óbito) e 4 medidas (`idade`, `dias_ate_obito`, `flag_obito` e `qtd_pessoas`). A saída foi salva em `data/intermediarios/fato.parquet`.

**Resultado:** 806.977 linhas e 16 colunas, na ordem do DDL da ETL-18.

**O que foi conferido:**
- **Totais:** 806.977 pessoas, 402.695 sinistros e 15.793 óbitos, iguais aos da base limpa. Por ano, 260.886, 266.128 e 279.963 pessoas, iguais aos da fonte.
- **Chaves:** as 9 chaves existem nas dimensões e nenhuma é nula. Nenhuma pessoa ficou sem chave: as funções de busca param com erro em vez de mandar um valor inesperado para o -1.
- **Regras do DDL:** valem todas as `CHECK` da tabela, inclusive a coerência do óbito. Quem não morreu tem data e local do óbito -2 e `dias_ate_obito` nulo.
- **Medidas:** `dias_ate_obito` é igual à diferença entre as duas datas em todos os óbitos.
- **Ida e volta:** ligando a fato às dimensões, cada pessoa recupera exatamente os valores da base limpa (gravidade, perfil, transporte, via, município, profissão, local e datas).

**Membros especiais:** o -2 aparece só onde o dado não se aplica: 791.184 pessoas sem óbito, na data e no local do óbito. O -1 aparece onde o dado faltou na fonte, por exemplo nas 70.314 pessoas sem gravidade e nas 86.834 sem profissão.

**Entradas:** cada grupo de colunas vem do mesmo arquivo que gerou a sua dimensão (ETL-13 a ETL-17), juntado pelo `id_pessoa`. Na execução final (ETL-26), todos apontam para o mesmo arquivo.

**Próximo passo:** a ETL-24 carrega as 8 dimensões e esta fato no PostgreSQL, nessa ordem, por causa das chaves estrangeiras.

---

## Etapa 09 · Carga no PostgreSQL (ETL-24)

- **Etapa:** ETL-24, Carga no PostgreSQL (Supabase)
- **Responsável:** Pedro Wellington
- **Issue:** ETL-24
- **Entrada:** `sql/ddl/etl_star_schema.sql`, `data/intermediarios/dims/*.parquet` e `data/intermediarios/fato.parquet`
- **Saída:** as 9 tabelas do modelo estrela no banco

Cria as tabelas pelo DDL e carrega as 8 dimensões e depois a fato. Pode rodar mais de uma vez: cada execução recria as tabelas e carrega do zero.

### 1. Setup

In [120]:
DDL    = RAIZ / "sql" / "ddl" / "etl_star_schema.sql"
FATO   = INTERM / "fato.parquet"

# ordem de carga: as dimensões primeiro, a fato por último (ela aponta para as dimensões)
NOMES_DIMS = ['dim_tempo', 'dim_gravidade', 'dim_perfil_vitima', 'dim_transporte',
              'dim_localidade', 'dim_via', 'dim_profissao', 'dim_local_obito']

testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


True

### 2. Criar as tabelas pelo DDL

In [121]:
engine = obter_engine()

# lê o script e executa tudo de uma vez: ele apaga e recria as 9 tabelas
sql = DDL.read_text(encoding='utf-8')
conn = engine.raw_connection()
try:
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.commit()
finally:
    conn.close()

# confere o que existe no banco agora
pd.read_sql("SELECT table_name FROM information_schema.tables "
            "WHERE table_schema = 'public' ORDER BY table_name", engine)

,table_name
0,dim_gravidade
1,dim_local_obito
2,dim_localidade
3,dim_perfil_vitima
4,dim_profissao
5,dim_tempo
6,dim_transporte
7,dim_via
8,fato_pessoa_sinistro


### 3. Carregar as tabelas

In [122]:
def ajustar_inteiros(df: pd.DataFrame) -> pd.DataFrame:
    """Colunas float que só têm números inteiros (ex.: 12.0) viram inteiro, senão o banco recusa."""
    df = df.copy()
    for c in df.select_dtypes(include='float').columns:
        if (df[c].dropna() % 1 == 0).all():
            df[c] = df[c].astype('Int64')
    return df

def carregar_tabela(nome: str, df: pd.DataFrame, lote: int = 100_000) -> None:
    """Envia o DataFrame para a tabela com COPY, em lotes de 100 mil linhas."""
    df = ajustar_inteiros(df)
    colunas = ', '.join(df.columns)
    conn = engine.raw_connection()
    try:
        with conn.cursor() as cur:
            for i in range(0, len(df), lote):
                buf = io.StringIO()
                df.iloc[i:i + lote].to_csv(buf, index=False, header=False)   # lote em CSV, na memória
                buf.seek(0)
                cur.copy_expert(f"COPY {nome} ({colunas}) FROM STDIN WITH (FORMAT csv)", buf)
        conn.commit()
    finally:
        conn.close()

In [123]:
# dimensões primeiro, a fato por último; cada tabela é conferida contra o parquet
for nome in NOMES_DIMS + ['fato_pessoa_sinistro']:
    arquivo = FATO if nome == 'fato_pessoa_sinistro' else DIMS / f'{nome}.parquet'
    df = pd.read_parquet(arquivo)
    carregar_tabela(nome, df)
    n_banco = pd.read_sql(f'SELECT COUNT(*) AS n FROM {nome}', engine)['n'][0]
    print(f'{nome:24} parquet: {len(df):>8} | banco: {n_banco:>8} | ok: {len(df) == n_banco}')

dim_tempo                parquet:     1121 | banco:     1121 | ok: True


dim_gravidade            parquet:        6 | banco:        6 | ok: True


dim_perfil_vitima        parquet:     1500 | banco:     1500 | ok: True


dim_transporte           parquet:       25 | banco:       25 | ok: True


dim_localidade           parquet:      647 | banco:      647 | ok: True


dim_via                  parquet:       10 | banco:       10 | ok: True


dim_profissao            parquet:      561 | banco:      561 | ok: True


dim_local_obito          parquet:        5 | banco:        5 | ok: True


fato_pessoa_sinistro     parquet:   806977 | banco:   806977 | ok: True


### 4. Resumo

**O que foi feito:** criação das 9 tabelas do modelo estrela no PostgreSQL (Supabase) pelo DDL da ETL-18 e carga das 8 dimensões e da fato.

**Como foi carregado:** `COPY` em lotes de 100 mil linhas, a partir dos parquets. As dimensões entram primeiro e a fato por último, porque ela aponta para as chaves delas.

**Rodar mais de uma vez:** o DDL começa apagando e recriando as tabelas, então cada execução carrega do zero e não duplica nada.

**Resultado:**

| Tabela | Linhas |
|---|---|
| dim_tempo | 1.121 |
| dim_gravidade | 6 |
| dim_perfil_vitima | 1.500 |
| dim_transporte | 25 |
| dim_localidade | 647 |
| dim_via | 10 |
| dim_profissao | 561 |
| dim_local_obito | 5 |
| fato_pessoa_sinistro | 806.977 |

**O que foi conferido:** a contagem de cada tabela no banco bate com a do parquet de origem.

**Observação:** colunas que vinham como float com valores inteiros (por exemplo 12.0) são convertidas antes da carga, senão o banco recusa. A senha do banco fica só no `.env`, que não vai para o Git.

---

## Etapa 10 · Validação pós-carga (ETL-25)

- **Etapa:** ETL-25, Validação pós-carga
- **Responsável:** Pedro Wellington
- **Issue:** ETL-25
- **Entrada:** as 9 tabelas carregadas no banco (ETL-24)
- **Saída:** `sql/validacao/etl_checks.sql`

Confere o banco depois da carga: contagens por tabela, chaves estrangeiras sem correspondência, total de óbitos e três consultas de exemplo.

### 1. Setup

In [124]:
engine = obter_engine()
SAIDA_SQL = RAIZ / "sql" / "validacao" / "etl_checks.sql"   # arquivo com as consultas
SAIDA_SQL.parent.mkdir(parents=True, exist_ok=True)

CONSULTAS = {}   # nome -> texto SQL, para gravar no arquivo no final

def consultar(nome: str, sql: str) -> pd.DataFrame:
    """Roda a consulta no banco e guarda o texto para o arquivo etl_checks.sql."""
    CONSULTAS[nome] = sql.strip()
    return pd.read_sql(sql, engine)

_ = testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


### 2. Contagens por tabela

In [125]:
consultar('contagem_por_tabela', """
SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM dim_tempo
UNION ALL SELECT 'dim_gravidade', COUNT(*) FROM dim_gravidade
UNION ALL SELECT 'dim_perfil_vitima', COUNT(*) FROM dim_perfil_vitima
UNION ALL SELECT 'dim_transporte', COUNT(*) FROM dim_transporte
UNION ALL SELECT 'dim_localidade', COUNT(*) FROM dim_localidade
UNION ALL SELECT 'dim_via', COUNT(*) FROM dim_via
UNION ALL SELECT 'dim_profissao', COUNT(*) FROM dim_profissao
UNION ALL SELECT 'dim_local_obito', COUNT(*) FROM dim_local_obito
UNION ALL SELECT 'fato_pessoa_sinistro', COUNT(*) FROM fato_pessoa_sinistro
""")

,tabela,linhas
0,dim_tempo,1121
1,dim_gravidade,6
2,dim_perfil_vitima,1500
3,dim_transporte,25
4,dim_localidade,647
5,dim_via,10
6,dim_profissao,561
7,dim_local_obito,5
8,fato_pessoa_sinistro,806977


### 3. Chaves estrangeiras sem correspondência

In [126]:
consultar('fk_sem_correspondencia', """
SELECT 'dim_tempo (data_sinistro)' AS chave, COUNT(*) AS orfaos
FROM fato_pessoa_sinistro f LEFT JOIN dim_tempo d ON f.sk_data_sinistro = d.sk_tempo
WHERE d.sk_tempo IS NULL
UNION ALL SELECT 'dim_tempo (data_obito)', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_tempo d ON f.sk_data_obito = d.sk_tempo
WHERE d.sk_tempo IS NULL
UNION ALL SELECT 'dim_gravidade', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_gravidade d ON f.sk_gravidade = d.sk_gravidade
WHERE d.sk_gravidade IS NULL
UNION ALL SELECT 'dim_perfil_vitima', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_perfil_vitima d ON f.sk_perfil_vitima = d.sk_perfil_vitima
WHERE d.sk_perfil_vitima IS NULL
UNION ALL SELECT 'dim_transporte', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_transporte d ON f.sk_transporte = d.sk_transporte
WHERE d.sk_transporte IS NULL
UNION ALL SELECT 'dim_localidade', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_localidade d ON f.sk_localidade = d.sk_localidade
WHERE d.sk_localidade IS NULL
UNION ALL SELECT 'dim_via', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_via d ON f.sk_via = d.sk_via
WHERE d.sk_via IS NULL
UNION ALL SELECT 'dim_profissao', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_profissao d ON f.sk_profissao = d.sk_profissao
WHERE d.sk_profissao IS NULL
UNION ALL SELECT 'dim_local_obito', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_local_obito d ON f.sk_local_obito = d.sk_local_obito
WHERE d.sk_local_obito IS NULL
""")

,chave,orfaos
0,dim_tempo (data_sinistro),0
1,dim_tempo (data_obito),0
2,dim_gravidade,0
3,dim_perfil_vitima,0
4,dim_transporte,0
5,dim_localidade,0
6,dim_via,0
7,dim_profissao,0
8,dim_local_obito,0


### 4. Total de óbitos

In [127]:
consultar('total_obitos', """
SELECT SUM(flag_obito) AS obitos, COUNT(*) AS vitimas
FROM fato_pessoa_sinistro
""")

,obitos,vitimas
0,15793,806977


### 5. Consultas de exemplo

Três consultas que juntam a fato com as dimensões, para conferir que os relacionamentos funcionam.

In [128]:
consultar('obitos_por_ano', """
SELECT t.ano, SUM(f.flag_obito) AS obitos, COUNT(*) AS vitimas
FROM fato_pessoa_sinistro f
JOIN dim_tempo t ON f.sk_data_sinistro = t.sk_tempo
GROUP BY t.ano
ORDER BY t.ano
""")

,ano,obitos,vitimas
0,2021,4858,260886
1,2022,5445,266128
2,2023,5490,279963


In [129]:
consultar('top10_municipios_obitos', """
SELECT l.municipio, SUM(f.flag_obito) AS obitos
FROM fato_pessoa_sinistro f
JOIN dim_localidade l ON f.sk_localidade = l.sk_localidade
GROUP BY l.municipio
ORDER BY obitos DESC
LIMIT 10
""")

,municipio,obitos
0,SAO PAULO,2525
1,CAMPINAS,445
2,GUARULHOS,423
3,SAO BERNARDO DO CAMPO,282
4,RIBEIRAO PRETO,268
5,SOROCABA,265
6,SAO JOSE DOS CAMPOS,218
7,JUNDIAI,211
8,MOGI DAS CRUZES,167
9,OSASCO,167


In [130]:
consultar('vitimas_por_sexo_e_tipo', """
SELECT p.sexo, tr.tipo_de_vitima, COUNT(*) AS vitimas, SUM(f.flag_obito) AS obitos
FROM fato_pessoa_sinistro f
JOIN dim_perfil_vitima p ON f.sk_perfil_vitima = p.sk_perfil_vitima
JOIN dim_transporte tr ON f.sk_transporte = tr.sk_transporte
GROUP BY p.sexo, tr.tipo_de_vitima
ORDER BY vitimas DESC
""")

,sexo,tipo_de_vitima,vitimas,obitos
0,MASCULINO,CONDUTOR,297070,9079
1,MASCULINO,PASSAGEIRO,182574,785
2,FEMININO,PASSAGEIRO,88746,969
3,FEMININO,CONDUTOR,79557,721
4,MASCULINO,NÃO INFORMADO,74699,500
5,FEMININO,NÃO INFORMADO,33999,110
6,MASCULINO,PEDESTRE,26104,2742
7,FEMININO,PEDESTRE,19862,821
8,NÃO INFORMADO,NÃO INFORMADO,4130,7
9,NÃO INFORMADO,PEDESTRE,207,31


### 6. Gravar o arquivo SQL

In [131]:
cabecalho = (
    "-- Consultas de validação pós-carga (ETL-25)\n"
    "-- Geradas pelo notebook notebooks/etl/10_validacao.ipynb\n\n"
)
blocos = [f"-- {nome}\n{sql};\n" for nome, sql in CONSULTAS.items()]

SAIDA_SQL.write_text(cabecalho + "\n".join(blocos), encoding="utf-8")
print(f"{len(CONSULTAS)} consultas gravadas em {SAIDA_SQL.relative_to(RAIZ)}")

6 consultas gravadas em sql\validacao\etl_checks.sql


### 7. Resumo

- As 9 tabelas estão no banco com as contagens esperadas: 8 dimensões e a fato com 806.977 linhas.
- Nenhuma chave estrangeira da fato ficou sem correspondência nas dimensões (0 em todas as 9 chaves).
- O total de óbitos na fato é 15.793, igual ao da base original.
- Foram escritas 3 consultas de exemplo juntando a fato com as dimensões.
- As consultas ficam salvas em `sql/validacao/etl_checks.sql`.